# SIH26166 — OHRC ↔ LRO NAC registration (**v4**, SuperGlue-first, no radiometric calibration)

Team Code4Bharat · derived from `sih_final_v3_2_no_isis.ipynb`.

## What this notebook does

Registers a Chandrayaan-2 **OHRC** strip (source) against an LRO **NAC** strip (reference),
classically and deterministically: crop to the shared footprint → resample both to a common
ground sampling distance → resolve orientation → **SuperPoint + SuperGlue** coarse fit →
tiled fine fit at native resolution → guarded model → optional ECC → honest metrics.

Homographies map **OHRC pixels → NAC pixels** throughout.

## What changed from v3.2

**Bugs fixed**

| # | Defect in v3.2 | Effect |
|---|---|---|
| C1 | `robust_fit()` has no `min_axis_ratio` parameter, but cell 12 passes one | **`TypeError` — the run dies right after the fine stage, before any result** |
| C2 | Longitude degrees converted to km with the latitude constant (30.33 km/deg) | Every across-track distance reported 2× too large at this pair's ~60 °N; `PAD_DEG` silently padded twice as far along-track as across-track |
| C3 | No longitude branch unwrapping (the v3 changelog claims it; the code does not have it) | A pair straddling the prime meridian reads as ~355° apart and is rejected outright |
| C4 | Error message refers to `MATCHER_POOLING`, which is never defined | Sends you looking for a switch that does not exist |
| C5 | `to_uint8` percentiles computed over warp-border zeros | A tile that is 40 % black border has its real range flattened before the matcher sees it |

**Preprocessing and matching**

- **The ORB flip test is replaced** by a dense ZNCC search on gradient magnitude, with a
  phase-randomised surrogate as an independent null. Validated on synthetic pairs with known
  flips and a 35° sun-angle change, on the real preview strips, and against three negative
  controls. It commits to one orientation when the evidence is clear (4× less coarse work)
  and falls back to all four when it is not. **It is advisory and never raises** — a gate that
  halts a correct run is worse than no gate.
- **RIFT2 is off by default** (`USE_RIFT2 = False`). No git clone, no CPU phase congruency.
  Flip it on to run the ablation; the per-matcher inlier breakdown is reported either way.
- **SuperGlue input size is per call** — 640 px tiles are no longer upsampled to 1024.
- **Dead tiles are screened out** on a Laplacian texture score before the GPU is asked to
  look at them. At 81° incidence a lot of tiles are pure shadow.

**Deliberately NOT here: radiometric calibration.** No decompanding, no dark/flat, no I/F,
no ISIS. Every appearance step in this notebook exists to make two images *comparable to a
matcher*, not to put them in physical units. If you later want the one radiometric step that
actually matters for matching, it is decompanding (8-bit companded → 12-bit linear), because
it is the only nonlinear one.

## Before the first run

1. **OHRC** — nothing to do; corners, dimensions and GSD come from the PDS4 `.xml` label.
2. **NAC** — EDR labels carry no footprint. Add a row to `KNOWN_NAC_CORNERS` in the config
   cell from the product page at <https://data.lroc.im-ldi.com/lroc/search>.
3. Read the banner the config cell prints. It names the pair actually in use.
4. `Runtime → Change runtime type → GPU`. SuperGlue on CPU is slow but will run.

In [ ]:
# 1) Mount Drive and install. RIFT2's requirements are NOT installed unless you enable it.
from google.colab import drive
drive.mount('/content/drive')

!pip install -q opencv-contrib-python planetaryimage pvl scikit-image scikit-learn
print("\nIf cv2 was already imported by something else this session, do "
      "Runtime -> Restart session once, then run from the top.")

In [ ]:
# 2) Config + pair registry
import os, json, glob, shutil, zipfile, time

PROJECT_ROOT = "/content/drive/MyDrive/SIH26166"
DATA_RAW     = os.path.join(PROJECT_ROOT, "data", "raw")
NAC_DIR      = os.path.join(DATA_RAW, "nac")
OHRC_DIR     = os.path.join(DATA_RAW, "ohrc")
OUT_DIR      = os.path.join(PROJECT_ROOT, "outputs")
PAIRS_JSON   = os.path.join(PROJECT_ROOT, "data", "pairs.json")
os.makedirs(OUT_DIR, exist_ok=True)

PAIR_ID = "pair01_equatorial"

# ---- run switches -----------------------------------------------------------------------
USE_RIFT2             = False   # True adds RIFT2 alongside SuperGlue (slow, CPU-bound)
ALLOW_TRUNCATED_OHRC  = False   # True runs on a partial upload as a plumbing test
RUN_ECC               = True    # gated intensity refinement after the tiled fit
# SuperGlue in the TILED stage. Measured on this pair (OHRC 28.3 deg sun elevation vs NAC
# 8.4 deg): SuperGlue contributed 0 of 355 correspondences across ~250 tiles, and caused
# 272 of the no-consensus rejections. At low sun the fine-scale texture is shadow edges
# with no counterpart in the other image, so SuperPoint fires on features that are not
# there. Dense correlation carried the entire result. Set True to re-measure on a new pair.
USE_SUPERGLUE_IN_TILES = False
MAX_TILES             = 400     # time guard on the fine stage (350 positions on
                                # this pair — 200 only covered the first 57%)

# Skip directories that hold derived products, never raw input.
_SKIP_DIRS = {"calibrated", ".ipynb_checkpoints", "__pycache__"}

def _walk(root):
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d.lower() not in _SKIP_DIRS]
        for fn in filenames:
            yield os.path.join(dirpath, fn)

def find_first(root, exts, recursive=True):
    exts = {e.lower() for e in exts}
    src = _walk(root) if recursive else (
        os.path.join(root, f) for f in sorted(os.listdir(root))
        if os.path.isfile(os.path.join(root, f)))
    for p in sorted(src):
        if os.path.splitext(p)[1].lower() in exts:
            return p
    return None

# ---- NAC: canonical folder, non-recursive, 'calibrated/' excluded ------------------------
nac_canonical_dir = os.path.join(NAC_DIR, PAIR_ID)
os.makedirs(nac_canonical_dir, exist_ok=True)
nac_img = find_first(nac_canonical_dir, [".img"], recursive=False)

if nac_img is None:                       # adopt a stray pairNN/ only if canonical is empty
    for stray in sorted(glob.glob(os.path.join(NAC_DIR, "pair*"))):
        if os.path.basename(stray) == PAIR_ID or not os.path.isdir(stray):
            continue
        hit = find_first(stray, [".img"], recursive=False)
        if hit:
            dest = os.path.join(nac_canonical_dir, os.path.basename(hit))
            print(f"Moving {hit} -> {dest}")
            shutil.move(hit, dest); nac_img = dest
            try: os.rmdir(stray)
            except OSError: pass
            break

if nac_img is None:
    raise FileNotFoundError(f"No NAC .IMG under {nac_canonical_dir}.")

nac_product_id = os.path.splitext(os.path.basename(nac_img))[0]

for stray in sorted(glob.glob(os.path.join(NAC_DIR, "pair*"))):
    if os.path.basename(stray) == PAIR_ID or not os.path.isdir(stray):
        continue
    left = [f for f in os.listdir(stray) if f.lower().endswith(".img")]
    if left:
        print(f"\n*** NOTE: {stray} still holds {left}, which this run IGNORES. ***")

# ---- OHRC: recursive, auto-unzip ---------------------------------------------------------
ohrc_canonical_dir = os.path.join(OHRC_DIR, PAIR_ID)
ohrc_img = find_first(ohrc_canonical_dir, [".img", ".qub"])
if ohrc_img is None:
    z = find_first(ohrc_canonical_dir, [".zip"])
    if z:
        print(f"Extracting {z}")
        with zipfile.ZipFile(z) as zf: zf.extractall(os.path.dirname(z))
        ohrc_img = find_first(ohrc_canonical_dir, [".img", ".qub"])
ohrc_xml = find_first(ohrc_canonical_dir, [".xml"])
if ohrc_img is None or ohrc_xml is None:
    print("\nUnder", ohrc_canonical_dir, ":")
    for f in _walk(ohrc_canonical_dir): print("  -", os.path.relpath(f, ohrc_canonical_dir))
    raise FileNotFoundError(f"Need an OHRC raster and .xml — found img={ohrc_img}, xml={ohrc_xml}.")
ohrc_product_id = os.path.splitext(os.path.basename(ohrc_img))[0]

# -----------------------------------------------------------------------------------------
# NAC footprint corners — the ONLY source, since EDR labels carry no footprint.
# ADD A PAIR BY ADDING A ROW. Never edit a row in place for a different product.
# Values are (north_lat, south_lat, west_lon, east_lon), degrees, +East.
#
# NOTE: isda:area = "Equatorial" in an OHRC label is a PROJECTION BUCKET, not a latitude.
# It means "not polar stereographic" and covers roughly +/-80 deg. Do not infer a latitude
# from it — read the label's actual corner coordinates, which the loader prints.
# -----------------------------------------------------------------------------------------
KNOWN_NAC_CORNERS = {
    # product_id: {UL, UR, LL, LR} -> (lat, lon) exactly as the product page lists them
    "M177656091LE": {"UL": (61.61, 355.45), "UR": (61.60, 355.29),
                     "LL": (59.92, 355.53), "LR": (59.92, 355.38)},
    "M175124932RE": {"UL": ( 1.11,  23.49), "UR": ( 1.11,  23.45),
                     "LL": ( 0.15,  23.50), "LR": ( 0.15,  23.46)},
}

KNOWN_NAC_GSD = {
    "M177656091LE": 0.933909314468014,   # page: Resolution (width 0.89 / height 0.97)
    "M175124932RE": 0.39935843279144334,
}

def _corners_from_table(entry):
    """Accepts the 4-corner dict above. A legacy (N,S,W,E) tuple is refused rather than
    silently rebuilt into a rectangle -- that is the bug this fix exists to remove."""
    if isinstance(entry, dict) and set(entry) == {"UL", "UR", "LL", "LR"}:
        return {k: {"lat": float(v[0]), "lon": float(v[1])} for k, v in entry.items()}
    raise ValueError(
        "KNOWN_NAC_CORNERS entries must be a dict of the four real corners, e.g.\n"
        '  "M177656091LE": {"UL": (61.61, 355.45), "UR": (61.60, 355.29),\n'
        '                   "LL": (59.92, 355.53), "LR": (59.92, 355.38)}\n'
        "A (north, south, west, east) bounding box is NOT enough: a NAC strip is a slanted\n"
        "parallelogram, and the rectangle you get from a box is the wrong width and can be\n"
        "mirrored. Read all four corners off the LROC product page.")

_entry = KNOWN_NAC_CORNERS.get(nac_product_id)
if _entry is not None:
    nac_corners_dict = _corners_from_table(_entry)
    NAC_CORNERS_SOURCE = "KNOWN_NAC_CORNERS (four real corners)"
else:
    nac_corners_dict, NAC_CORNERS_SOURCE = None, None
    print(f"\n*** No corner entry for '{nac_product_id}'. Look it up at "
          f"https://data.lroc.im-ldi.com/lroc/search and copy ALL FOUR corners. ***")

registry = {}
if os.path.exists(PAIRS_JSON):
    try:
        with open(PAIRS_JSON) as f: registry = json.load(f)
    except json.JSONDecodeError:
        print("(pairs.json was unreadable — starting a fresh registry)")
registry[PAIR_ID] = {
    "nac":  {"product_id": nac_product_id, "path": nac_img,
             "corners_deg": nac_corners_dict, "corners_source": NAC_CORNERS_SOURCE},
    "ohrc": {"product_id": ohrc_product_id, "path": ohrc_img, "label_path": ohrc_xml},
}
with open(PAIRS_JSON, "w") as f: json.dump(registry, f, indent=2)

print("\n" + "=" * 74)
print(f"PAIR IN USE: {PAIR_ID}")
print(f"  OHRC : {ohrc_product_id}")
print(f"  NAC  : {nac_product_id}")
print(f"  NAC corners: {NAC_CORNERS_SOURCE or '*** MISSING — add a row above ***'}")
print(f"  RIFT2: {'ON' if USE_RIFT2 else 'OFF (SuperGlue only)'}")
print("=" * 74)

In [ ]:
# 3) Geodesy + transform algebra  [v4: cos(lat) and longitude branch handling are NEW]
#
# v3 converted BOTH lat and lon degrees to km with the same 30.33 km/deg constant. That is
# only true for latitude. A degree of longitude shrinks as cos(lat), so at this pair's ~60 N
# every across-track distance v3 printed was 2x too large, and PAD_DEG added twice as much
# padding along-track as across-track without saying so.
import numpy as np, cv2

MOON_R_KM      = 1737.4
KM_PER_DEG_LAT = 2.0 * np.pi * MOON_R_KM / 360.0        # 30.33 km/deg, latitude only

def km_per_deg_lon(lat_deg):
    """Kilometres per degree of longitude at a given latitude."""
    return KM_PER_DEG_LAT * float(np.cos(np.radians(lat_deg)))

def unwrap_lon(lon, ref):
    """Put `lon` on the same 360-degree branch as `ref`.

    Without this, a product at 355.5 E and one at 0.5 E read as 355 degrees apart instead
    of 5, and a pair straddling the prime meridian is rejected as non-overlapping.
    """
    lon = np.asarray(lon, dtype=np.float64)
    return lon - 360.0 * np.round((lon - float(ref)) / 360.0)

def ground_extent_km(lat_min, lat_max, lon_min, lon_max):
    """(along-track km, across-track km) for a lat/lon box, honest about cos(lat)."""
    mid_lat = 0.5 * (lat_min + lat_max)
    return ((lat_max - lat_min) * KM_PER_DEG_LAT,
            (lon_max - lon_min) * km_per_deg_lon(mid_lat))

# --- pixel <-> lat/lon, affine through the 4 corners ------------------------------------
def fit_affine_pixel_to_latlon(corners_deg, shape):
    """corners_deg: {'UL','UR','LL','LR'} -> {'lat','lon'}.  shape: (n_lines, n_samples).

    Exact at the corners, a mild linear approximation between them. Longitudes are
    unwrapped onto the UL corner's branch first so a prime-meridian crossing fits cleanly.
    """
    n_lines, n_samples = shape
    pix = np.array([[0, 0], [0, n_samples - 1],
                    [n_lines - 1, 0], [n_lines - 1, n_samples - 1]], dtype=np.float64)
    keys = ("UL", "UR", "LL", "LR")
    lat = np.array([corners_deg[k]["lat"] for k in keys], dtype=np.float64)
    lon_ref = float(corners_deg["UL"]["lon"])
    lon = np.array([float(unwrap_lon(corners_deg[k]["lon"], lon_ref)) for k in keys])

    A = np.column_stack([pix[:, 0], pix[:, 1], np.ones(4)])
    coef_lat, *_ = np.linalg.lstsq(A, lat, rcond=None)
    coef_lon, *_ = np.linalg.lstsq(A, lon, rcond=None)
    M = np.array([coef_lat[:2], coef_lon[:2]])
    t = np.array([coef_lat[2], coef_lon[2]])
    if abs(np.linalg.det(M)) < 1e-15:
        raise ValueError("Corner coordinates are degenerate — the 4 corners do not span "
                         "a 2D patch. Check the KNOWN_NAC_CORNERS row for this product.")
    M_inv = np.linalg.inv(M)

    def forward(row, col):
        return tuple(M @ np.array([row, col], dtype=np.float64) + t)

    def inverse(lat_, lon_):
        lon_u = float(unwrap_lon(lon_, lon_ref))
        row, col = M_inv @ (np.array([lat_, lon_u], dtype=np.float64) - t)
        return row, col

    return forward, inverse

# --- 3x3 transform helpers.  Convention: points are (x=col, y=row), q = H @ p ------------
def mat_translate(tx, ty):
    return np.array([[1.0, 0.0, tx], [0.0, 1.0, ty], [0.0, 0.0, 1.0]])

def mat_scale(sx, sy=None):
    sy = sx if sy is None else sy
    return np.array([[sx, 0.0, 0.0], [0.0, sy, 0.0], [0.0, 0.0, 1.0]])

def to_h(M23):
    return np.vstack([np.asarray(M23, dtype=np.float64), [0.0, 0.0, 1.0]])

def apply_h(H, pts):
    pts = np.asarray(pts, dtype=np.float64).reshape(-1, 2)
    if len(pts) == 0:
        return pts
    p = np.hstack([pts, np.ones((len(pts), 1))])
    q = (np.asarray(H, dtype=np.float64) @ p.T).T
    return q[:, :2] / q[:, 2:3]

def H_at_level(H, gsd_from, gsd_to):
    """Re-express a transform measured at one ground sampling distance at another."""
    k = gsd_from / gsd_to
    return mat_scale(k) @ np.asarray(H, dtype=np.float64) @ mat_scale(1.0 / k)

def mat_flip(flip_code, h, w):
    """3x3 equivalent of cv2.flip(img, flip_code) on an image of shape (h, w)."""
    if flip_code is None:
        return np.eye(3)
    fx = -1.0 if flip_code in (1, -1) else 1.0
    fy = -1.0 if flip_code in (0, -1) else 1.0
    tx = (w - 1) if fx < 0 else 0.0
    ty = (h - 1) if fy < 0 else 0.0
    return np.array([[fx, 0.0, tx], [0.0, fy, ty], [0.0, 0.0, 1.0]])

def mat_rotate_about(center_xy, angle_deg):
    return to_h(cv2.getRotationMatrix2D(tuple(map(float, center_xy)), float(angle_deg), 1.0))

print("Geodesy + transform helpers ready.")
print(f"  latitude  : {KM_PER_DEG_LAT:.3f} km/deg (constant)")
for _lat in (0, 30, 60):
    print(f"  longitude at {_lat:2d} N: {km_per_deg_lon(_lat):.3f} km/deg")

In [ ]:
# 4) OHRC (PDS4) loader
import xml.etree.ElementTree as ET

NS = {"pds": "http://pds.nasa.gov/pds4/pds/v1",
      "isda": "https://isda.issdc.gov.in/pds4/isda/v1"}

def load_ohrc(img_path, xml_path):
    root = ET.parse(xml_path).getroot()
    axes = root.findall(".//pds:Array_2D_Image/pds:Axis_Array", NS)
    dims = {a.find("pds:axis_name", NS).text: int(a.find("pds:elements", NS).text) for a in axes}
    n_lines, n_samples = dims["Line"], dims["Sample"]

    geom = root.find(".//isda:Refined_Corner_Coordinates", NS)
    if geom is None:
        raise ValueError("No isda:Refined_Corner_Coordinates in the OHRC label.")
    corners_deg = {k: {"lat": float(geom.find(f"isda:{t}_latitude", NS).text),
                       "lon": float(geom.find(f"isda:{t}_longitude", NS).text)}
                   for k, t in (("UL", "upper_left"), ("UR", "upper_right"),
                                ("LL", "lower_left"), ("LR", "lower_right"))}

    area = root.find(".//isda:area", NS)
    proj = root.find(".//isda:projection", NS)
    lvl  = root.find(".//pds:Primary_Result_Summary/pds:processing_level", NS)
    print(f"OHRC {n_lines} x {n_samples}   area={area.text if area is not None else '?'}"
          f"   projection={proj.text if proj is not None else '?'}"
          f"   level={lvl.text if lvl is not None else '?'}")
    print("  reminder: area='Equatorial' is a projection bucket (non-polar), NOT a latitude.")

    gsd = None
    for el in root.iter():
        tag = el.tag.split('}')[-1].lower()
        if ('resolution' in tag or 'pixel_size' in tag or 'sampling' in tag) and el.text:
            try: v = float(el.text.strip().split()[0])
            except ValueError: continue
            if gsd is None and 0.05 < v < 5.0:
                gsd = v
                print(f"  GSD from label field '{tag}': {v}")

    expected, actual = n_lines * n_samples, os.path.getsize(img_path)
    if actual < expected:
        avail = actual // n_samples
        msg = (f"{os.path.basename(img_path)} is {actual:,} bytes, label implies {expected:,} "
               f"— only {avail:,} of {n_lines:,} lines present ({100.0*actual/expected:.2f}%).")
        if not ALLOW_TRUNCATED_OHRC:
            raise ValueError(msg + "  Re-upload with Drive for Desktop (the browser upload "
                             "does not resume), or set ALLOW_TRUNCATED_OHRC = True.")
        print(f"\n*** {msg}  Running on the fragment — this is a smoke test, not a result. ***")
        frac = avail / float(n_lines)
        for top, bot in (("UL", "LL"), ("UR", "LR")):
            for k in ("lat", "lon"):
                corners_deg[bot][k] = corners_deg[top][k] + frac * (corners_deg[bot][k] -
                                                                   corners_deg[top][k])
        n_lines = avail
    else:
        print(f"  file size OK: {actual:,} bytes == {n_lines} x {n_samples}")

    raster = np.memmap(img_path, dtype=np.uint8, mode="r", shape=(n_lines, n_samples))
    return raster, corners_deg, (n_lines, n_samples), gsd

ohrc_raster, ohrc_corners, ohrc_shape, OHRC_GSD_FROM_LABEL = load_ohrc(ohrc_img, ohrc_xml)
ohrc_fwd, ohrc_inv = fit_affine_pixel_to_latlon(ohrc_corners, ohrc_shape)
print("\nOHRC corners from the label (this is the real footprint):")
for k, v in ohrc_corners.items():
    print(f"   {k}: {v['lat']:9.5f} N, {v['lon']:10.5f} E")
_al, _ac = ground_extent_km(min(c['lat'] for c in ohrc_corners.values()),
                            max(c['lat'] for c in ohrc_corners.values()),
                            min(c['lon'] for c in ohrc_corners.values()),
                            max(c['lon'] for c in ohrc_corners.values()))
print(f"   ground extent: {_al:.2f} km along-track x {_ac:.2f} km across-track")

In [ ]:
# 5) NAC (PDS3) loader
# The int8 -> uint8 reinterpretation is kept from v3: LROC NAC EDR DNs use the full 0-255
# range, so under int8 every sample above 127 reads negative and the percentile stretch runs
# on a wrapped histogram. This is a NumPy dtype bug, not a calibration step.
for _alias, _real in (("product", np.prod), ("cumproduct", np.cumprod),
                      ("alltrue", np.all), ("sometrue", np.any)):
    if not hasattr(np, _alias):
        setattr(np, _alias, _real)

from planetaryimage import PDS3Image

nac_pds  = PDS3Image.open(nac_img)
nac_full = np.asarray(nac_pds.image)
if nac_full.ndim == 3:
    nac_full = nac_full[0]
if nac_full.dtype == np.int8:
    print(f"NAC loaded as int8 (min={nac_full.min()}, max={nac_full.max()}) — "
          "reinterpreting as uint8.")
    nac_full = nac_full.view(np.uint8)
elif nac_full.dtype != np.uint8:
    print(f"NAC dtype is {nac_full.dtype} — check SAMPLE_TYPE before trusting the stretch.")
nac_shape = nac_full.shape

nac_corners = nac_corners_dict
if nac_corners is None:
    raise RuntimeError(
        f"NAC corners for '{nac_product_id}' are unknown.\n"
        f"  Add a row to KNOWN_NAC_CORNERS in the config cell (values from\n"
        f"  https://data.lroc.im-ldi.com/lroc/search), re-run it, then re-run this cell.")

nac_fwd, nac_inv = fit_affine_pixel_to_latlon(nac_corners, nac_shape)

NAC_GSD_FROM_LABEL = None
try:
    def _scan(obj, depth=0):
        global NAC_GSD_FROM_LABEL
        if depth > 3: return
        try: items = obj.items()
        except Exception: return
        for k, v in items:
            if 'RESOLUTION' in str(k).upper() or 'SCALED_PIXEL' in str(k).upper():
                try:
                    val = float(getattr(v, 'value', v))
                    if NAC_GSD_FROM_LABEL is None and 0.05 < val < 5.0:
                        NAC_GSD_FROM_LABEL = val
                        print(f"  GSD from label field '{k}': {val}")
                except (TypeError, ValueError): pass
            _scan(v, depth + 1)
    _scan(nac_pds.label)
except Exception as e:
    print("  (could not introspect the PDS3 label:", e, ")")

print(f"NAC raster {nac_full.shape} dtype={nac_full.dtype} "
      f"(min={int(nac_full.min())}, max={int(nac_full.max())})")
print("NAC corners in use:")
for k, v in nac_corners.items():
    print(f"   {k}: {v['lat']:9.5f} N, {v['lon']:10.5f} E")
_al, _ac = ground_extent_km(min(c['lat'] for c in nac_corners.values()),
                            max(c['lat'] for c in nac_corners.values()),
                            min(c['lon'] for c in nac_corners.values()),
                            max(c['lon'] for c in nac_corners.values()))
print(f"   ground extent: {_al:.2f} km along-track x {_ac:.2f} km across-track")

# --- Native ground sampling distances -------------------------------------------------
# Resolved HERE, not in the resolution cell, because the crop cell below pads in pixels
# (PAD_M / that image's own GSD) and cross-checks that both crops describe the same
# ground. Both of those need the GSDs before any cropping happens.
OHRC_NATIVE_GSD_M = OHRC_GSD_FROM_LABEL or 0.26
_g = KNOWN_NAC_GSD.get(nac_product_id)
NAC_NATIVE_GSD_M = _g or NAC_GSD_FROM_LABEL
if NAC_NATIVE_GSD_M is None:
    raise RuntimeError(
        f"No GSD for NAC '{nac_product_id}'. Add a KNOWN_NAC_GSD row from the product page's\n"
        f"  'Resolution' field. Never fall back to another product's value -- a wrong GSD\n"
        f"  silently breaks the common-GSD resampling and every match after it.")
print(f"NAC GSD {NAC_NATIVE_GSD_M:.4f} m/px "
      f"(from {'KNOWN_NAC_GSD' if _g else 'PDS3 label'})")

_m = np.array([[*nac_fwd(1, 0)], [*nac_fwd(0, 1)], [*nac_fwd(0, 0)]])
_dlat_row, _dlon_row = _m[0] - _m[2]
_dlat_col, _dlon_col = _m[1] - _m[2]
_midlat = np.mean([c["lat"] for c in nac_corners.values()])
_mpl = np.hypot(_dlat_row * KM_PER_DEG_LAT, _dlon_row * km_per_deg_lon(_midlat)) * 1000
_mps = np.hypot(_dlat_col * KM_PER_DEG_LAT, _dlon_col * km_per_deg_lon(_midlat)) * 1000
print(f"\nNAC corner-model self-check:")
print(f"  implied {_mpl:.3f} m/line, {_mps:.3f} m/sample   vs label GSD "
      f"{NAC_NATIVE_GSD_M:.3f} m")
print(f"  columns run {'EAST->WEST (mirrored strip — normal for NAC)' if _dlon_col < 0 else 'WEST->EAST'}")
print(f"  rows run {'NORTH->SOUTH' if _dlat_row < 0 else 'SOUTH->NORTH'}")
for _nm, _v in (("line", _mpl), ("sample", _mps)):
    _r = _v / NAC_NATIVE_GSD_M
    if not (0.85 < _r < 1.18):
        print(f"  *** {_nm} scale is x{_r:.2f} off the label GSD. The corners and the GSD\n"
              f"      disagree — most often a bounding box stored where four real corners\n"
              f"      were needed. Fix this before running anything downstream. ***")
    else:
        print(f"  {_nm} scale x{_r:.2f}  OK")
print(f"OHRC GSD {OHRC_NATIVE_GSD_M:.4f} m/px "
      f"({'label' if OHRC_GSD_FROM_LABEL else 'NOMINAL FALLBACK — verify'})")


In [ ]:
# 6) Overlap window and crop  [v4: cos(lat) + longitude unwrapping + metric padding]
#
# v3's PAD_DEG added the same number of DEGREES to latitude and longitude. At 60 N that is
# twice as much ground along-track as across-track. PAD_M is metres, converted per axis.
PAD_M = 150.0        # slack per side, matched to the affine corner model's own imprecision

def corner_bounds(corners_deg, lon_ref=None):
    lats = [c["lat"] for c in corners_deg.values()]
    ref  = corners_deg["UL"]["lon"] if lon_ref is None else lon_ref
    lons = [float(unwrap_lon(c["lon"], ref)) for c in corners_deg.values()]
    return min(lats), max(lats), min(lons), max(lons)

# Put BOTH products on the OHRC's longitude branch before any comparison.
LON_REF = ohrc_corners["UL"]["lon"]
o_lat0, o_lat1, o_lon0, o_lon1 = corner_bounds(ohrc_corners, LON_REF)
n_lat0, n_lat1, n_lon0, n_lon1 = corner_bounds(nac_corners,  LON_REF)

print(f"OHRC {ohrc_product_id}")
print(f"   lat {o_lat0:9.4f} .. {o_lat1:9.4f}   lon {o_lon0:10.4f} .. {o_lon1:10.4f}")
print(f"NAC  {nac_product_id}")
print(f"   lat {n_lat0:9.4f} .. {n_lat1:9.4f}   lon {n_lon0:10.4f} .. {n_lon1:10.4f}")

_ov_lat = min(o_lat1, n_lat1) - max(o_lat0, n_lat0)
_ov_lon = min(o_lon1, n_lon1) - max(o_lon0, n_lon0)
_mid_lat = 0.5 * (max(o_lat0, n_lat0) + min(o_lat1, n_lat1))

if _ov_lat <= 0 or _ov_lon <= 0:
    why = []
    if _ov_lat <= 0: why.append(f"{-_ov_lat*KM_PER_DEG_LAT:.2f} km apart in latitude")
    if _ov_lon <= 0: why.append(f"{-_ov_lon*km_per_deg_lon(_mid_lat):.2f} km apart in longitude")
    raise RuntimeError(
        "These two footprints do not overlap: " + " and ".join(why) + ".\n"
        "  Either the KNOWN_NAC_CORNERS row is wrong, or the two products image different\n"
        "  ground. Check both product IDs in the banner above. Note that an OHRC label\n"
        "  saying area='Equatorial' does NOT mean the scene is near the equator.")

print(f"\nRaw footprint overlap: {_ov_lat*KM_PER_DEG_LAT:.2f} km along-track "
      f"x {_ov_lon*km_per_deg_lon(_mid_lat):.2f} km across-track "
      f"(at {_mid_lat:.2f} N, 1 deg lon = {km_per_deg_lon(_mid_lat):.2f} km)")

# Which product limits each axis? The narrower swath drives the across-track crop; tiles cut
# outside it would match nothing.
print(f"  across-track limited by: "
      f"{'NAC' if (n_lon1-n_lon0) < (o_lon1-o_lon0) else 'OHRC'}")
print(f"  along-track  limited by: "
      f"{'NAC' if (n_lat1-n_lat0) < (o_lat1-o_lat0) else 'OHRC'}")

def _footprint_poly(corners, lon_ref):
    """(4,2) float32 polygon in (lon, lat), wound consistently."""
    order = ("UL", "UR", "LR", "LL")
    return np.array([[float(unwrap_lon(corners[k]["lon"], lon_ref)), corners[k]["lat"]]
                     for k in order], np.float32)

poly_o = _footprint_poly(ohrc_corners, LON_REF)
poly_n = _footprint_poly(nac_corners,  LON_REF)

_area, inter_poly = cv2.intersectConvexConvex(poly_o, poly_n)
USED_POLYGON_CROP = bool(_area > 0 and inter_poly is not None and len(inter_poly) >= 3)

if USED_POLYGON_CROP:
    inter_poly = np.asarray(inter_poly, np.float64).reshape(-1, 2)
    _ilat = inter_poly[:, 1]; _ilon = inter_poly[:, 0]
    _imid = 0.5 * (_ilat.min() + _ilat.max())
    print(f"\nFootprint POLYGON intersection: {len(inter_poly)} vertices, "
          f"{(_ilat.max()-_ilat.min())*KM_PER_DEG_LAT:.2f} km lat span x "
          f"{(_ilon.max()-_ilon.min())*km_per_deg_lon(_imid):.2f} km lon span")
else:
    print("\n*** Polygon intersection empty — falling back to the axis-aligned box. ***")
    _fl0, _fl1 = max(o_lat0, n_lat0), min(o_lat1, n_lat1)
    _fn0, _fn1 = max(o_lon0, n_lon0), min(o_lon1, n_lon1)
    inter_poly = np.array([[_fn0, _fl0], [_fn1, _fl0], [_fn1, _fl1], [_fn0, _fl1]])

def crop_to_polygon(raster, inverse_fn, verts_lonlat, gsd, name):
    """Crop to the pixel bounding box of a lat/lon polygon, padded by PAD_M in pixels."""
    n_lines, n_samples = raster.shape[:2]
    pix = [inverse_fn(float(v[1]), float(v[0])) for v in verts_lonlat]
    rows = [p[0] for p in pix]; cols = [p[1] for p in pix]
    # Pad in this image's own pixels, but never by more than 15% of the polygon's own
    # extent: a pad comparable to the overlap turns the crop into a no-op, which is the
    # whole failure this cell exists to prevent.
    _ext_r, _ext_c = max(rows) - min(rows), max(cols) - min(cols)
    pad_px = min(PAD_M / max(gsd, 1e-6), 0.15 * max(min(_ext_r, _ext_c), 1.0))
    r0 = int(max(0, np.floor(min(rows) - pad_px)))
    r1 = int(min(n_lines,   np.ceil(max(rows) + pad_px)))
    c0 = int(max(0, np.floor(min(cols) - pad_px)))
    c1 = int(min(n_samples, np.ceil(max(cols) + pad_px)))
    if r0 >= r1 or c0 >= c1:
        raise ValueError(f"{name}: computed crop window is empty — check the corner geometry.")
    print(f"  {name}: rows {r0}-{r1} ({r1-r0} of {n_lines}), cols {c0}-{c1} "
          f"({c1-c0} of {n_samples})   pad {pad_px:.0f} px")
    return raster[r0:r1, c0:c1], (r0, r1, c0, c1)

print("Cropping to the intersection polygon:")
ohrc_crop, ohrc_win = crop_to_polygon(ohrc_raster, ohrc_inv, inter_poly,
                                      OHRC_NATIVE_GSD_M, "OHRC")
nac_crop,  nac_win  = crop_to_polygon(nac_full, nac_inv, inter_poly,
                                      NAC_NATIVE_GSD_M, "NAC ")

def _kept(full, win):
    r0, r1, c0, c1 = win
    return 100.0 * ((r1 - r0) * (c1 - c0)) / (full[0] * full[1])

ohrc_kept, nac_kept = _kept(ohrc_shape, ohrc_win), _kept(nac_shape, nac_win)

# THE check that matters: after cropping to shared ground and before any resampling, the two
# crops must describe the SAME GROUND. If they do not, no matcher can rescue it.
_o_km = ((ohrc_win[1]-ohrc_win[0]) * OHRC_NATIVE_GSD_M / 1000.0,
         (ohrc_win[3]-ohrc_win[2]) * OHRC_NATIVE_GSD_M / 1000.0)
_n_km = ((nac_win[1]-nac_win[0]) * NAC_NATIVE_GSD_M / 1000.0,
         (nac_win[3]-nac_win[2]) * NAC_NATIVE_GSD_M / 1000.0)
print(f"\nOHRC crop {ohrc_crop.shape}  keeps {ohrc_kept:5.1f}%   "
      f"= {_o_km[0]:.2f} x {_o_km[1]:.2f} km of ground")
print(f"NAC  crop {nac_crop.shape}  keeps {nac_kept:5.1f}%   "
      f"= {_n_km[0]:.2f} x {_n_km[1]:.2f} km of ground")
_ra, _rc = _o_km[0]/max(_n_km[0],1e-9), _o_km[1]/max(_n_km[1],1e-9)
print(f"GROUND PARITY: along x{_ra:.2f}, across x{_rc:.2f}")
print( "  along-track should be ~1.0. ACROSS-track is routinely 1.2-1.5 even when correct:")
print( "  the two footprints are slanted parallelograms, so their intersection is slanted too,")
print( "  and an axis-aligned pixel bounding box around it carries margin on the wider image.")
print( "  What this check is really for is catching the gross failure -- a 2x or 3x ratio means")
print( "  a wrong GSD or a bounding box stored where four real corners belong.")
if not (0.70 < _ra < 1.45 and 0.60 < _rc < 1.75):
    print("*** The two crops do NOT describe the same ground. Do not continue — the coarse\n"
          "    stage cannot recover from this. Check, in order: the NAC GSD (FIX A), the\n"
          "    KNOWN_NAC_CORNERS row, and the KNOWN_NAC_CORNERS row. ***")
if ohrc_kept > 99.0 and nac_kept > 99.0:
    print("*** WARNING: the crop kept essentially both full rasters. ***")

T_ohrc_crop = mat_translate(-ohrc_win[2], -ohrc_win[0])
T_nac_crop  = mat_translate(-nac_win[2],  -nac_win[0])
import gc; gc.collect()


In [ ]:
# 7) Resolution scheme — two explicit levels
# (the native GSDs are resolved at the end of cell 5, because the crop cell needs them)
#   COARSE : whole overlap, small enough for a global orientation search
#   FINE   : never materialised; tiles cut from the native rasters on demand
print(f"native scale ratio NAC/OHRC = {NAC_NATIVE_GSD_M/OHRC_NATIVE_GSD_M:.3f}"
      "  <- this is what the common-GSD step removes")

# Matching scale. 1.0 = the coarser native GSD (never upsample). RAISE THIS when the pair
# correlates at coarse scale but not at native: at low sun elevation the fine-scale
# appearance is dominated by shadows that differ between the two acquisitions, while the
# real topography only emerges once those average out. The dense band check in the coarse
# cell tells you the scale at which the pair genuinely correlates.
FINE_GSD_MULTIPLIER = 1.0
TARGET_GSD_FINE_M   = max(OHRC_NATIVE_GSD_M, NAC_NATIVE_GSD_M) * FINE_GSD_MULTIPLIER
COARSE_PIXEL_BUDGET = 2_000_000

def _px_at(shape, native_gsd, gsd):
    return (shape[0] * native_gsd / gsd) * (shape[1] * native_gsd / gsd)

TARGET_GSD_COARSE_M = TARGET_GSD_FINE_M
while max(_px_at(ohrc_crop.shape, OHRC_NATIVE_GSD_M, TARGET_GSD_COARSE_M),
          _px_at(nac_crop.shape,  NAC_NATIVE_GSD_M,  TARGET_GSD_COARSE_M)) > COARSE_PIXEL_BUDGET:
    TARGET_GSD_COARSE_M *= 1.25

print(f"\nFINE   {TARGET_GSD_FINE_M:.3f} m/px (tiles only, multiplier {FINE_GSD_MULTIPLIER:g})")
print(f"COARSE {TARGET_GSD_COARSE_M:.3f} m/px (<= {COARSE_PIXEL_BUDGET/1e6:.1f} Mpx per image)")

def level_view(raster, native_gsd, target_gsd):
    f = native_gsd / target_gsd
    step = max(1, int(1.0 / f / 2)) if f < 1 else 1
    small = np.asarray(raster[::step, ::step])
    f2 = (native_gsd * step) / target_gsd
    if abs(f2 - 1.0) < 1e-6:
        return small
    return cv2.resize(small, None, fx=f2, fy=f2,
                      interpolation=cv2.INTER_AREA if f2 < 1 else cv2.INTER_LINEAR)

def level_tile(raster, native_gsd, target_gsd, r0, c0, h, w):
    """One tile at target_gsd. (r0,c0,h,w) in target-gsd px relative to the crop."""
    s = target_gsd / native_gsd
    R0, C0 = max(0, int(np.floor(r0 * s))), max(0, int(np.floor(c0 * s)))
    R1 = min(raster.shape[0], int(np.ceil((r0 + h) * s)))
    C1 = min(raster.shape[1], int(np.ceil((c0 + w) * s)))
    if R1 - R0 < 8 * s or C1 - C0 < 8 * s:
        return None, None
    patch = np.asarray(raster[R0:R1, C0:C1])
    oh, ow = max(1, int(round((R1 - R0) / s))), max(1, int(round((C1 - C0) / s)))
    return cv2.resize(patch, (ow, oh), interpolation=cv2.INTER_AREA), (C0 / s, R0 / s)

def level_shape(crop_shape, native_gsd, target_gsd):
    k = native_gsd / target_gsd
    return (int(round(crop_shape[0] * k)), int(round(crop_shape[1] * k)))

OHRC_FINE_SHAPE = level_shape(ohrc_crop.shape, OHRC_NATIVE_GSD_M, TARGET_GSD_FINE_M)
NAC_FINE_SHAPE  = level_shape(nac_crop.shape,  NAC_NATIVE_GSD_M,  TARGET_GSD_FINE_M)
M_ohrc_native_to_fine = mat_scale(OHRC_NATIVE_GSD_M / TARGET_GSD_FINE_M) @ T_ohrc_crop
M_nac_native_to_fine  = mat_scale(NAC_NATIVE_GSD_M  / TARGET_GSD_FINE_M) @ T_nac_crop

ohrc_coarse = level_view(ohrc_crop, OHRC_NATIVE_GSD_M, TARGET_GSD_COARSE_M)
nac_coarse  = level_view(nac_crop,  NAC_NATIVE_GSD_M,  TARGET_GSD_COARSE_M)
print(f"\nFine extents (virtual): OHRC {OHRC_FINE_SHAPE}, NAC {NAC_FINE_SHAPE}")
print(f"Coarse images:          OHRC {ohrc_coarse.shape}, NAC {nac_coarse.shape}")

In [ ]:
# 8) Appearance preprocessing  [v4: this is the whole radiometric story now]
#
# No decompanding, no dark/flat, no I/F. Every step here is about making two images
# COMPARABLE to a matcher, not about physical units:
#   to_uint8   - percentile stretch, robust to outliers and to nulls
#   CLAHE      - local contrast, which is what SuperPoint keys on
#   hist match - narrows the residual illumination gap between the two sensors
from skimage.exposure import match_histograms

def to_uint8(img, lo_p=1.0, hi_p=99.0, mask=None):
    """Percentile stretch to 8-bit.

    v4: `mask` (bool array, True = real data) replaces v3's implicit "use every pixel".
    A warped tile is part black border, and those zeros drag the low percentile down and
    flatten the real range. We do NOT guess the mask from `== 0`: in shadowed lunar terrain
    at high incidence angle a DN of 0 is legitimate data, so the caller passes the warp
    validity mask it already has and nothing is inferred.
    """
    a = np.asarray(img, dtype=np.float32)
    sel = np.isfinite(a)
    if mask is not None:
        sel &= np.asarray(mask, dtype=bool)
    finite = a[sel]
    if finite.size < 8:
        finite = a[np.isfinite(a)]
    if finite.size == 0:
        return np.zeros(a.shape, np.uint8)
    lo, hi = np.percentile(finite, lo_p), np.percentile(finite, hi_p)
    if hi <= lo:
        hi = lo + 1.0
    a = np.nan_to_num(a, nan=lo)
    return (np.clip((a - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)

def apply_clahe(img_u8, clip_limit, tile=(8, 8)):
    return cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile).apply(img_u8)

OHRC_CLIP_LIMIT        = 2.0
NAC_CLIP_LIMIT         = 4.0
USE_HISTOGRAM_MATCHING = True

def prep_pair(ohrc_raw, nac_raw, mask=None):
    """(ohrc_pc, nac_pc, ohrc_sg, nac_sg).

    *_pc  plain percentile stretch  -> RIFT2 / correlation / display
    *_sg  CLAHE (+ histogram match) -> SuperGlue
    """
    o_pc, n_pc = to_uint8(ohrc_raw, mask=mask), to_uint8(nac_raw)
    o_sg = apply_clahe(o_pc, OHRC_CLIP_LIMIT)
    n_sg = apply_clahe(n_pc, NAC_CLIP_LIMIT)
    if USE_HISTOGRAM_MATCHING:
        n_sg = np.clip(match_histograms(n_sg.astype(np.float32), o_sg.astype(np.float32)),
                       0, 255).astype(np.uint8)
    return o_pc, n_pc, o_sg, n_sg

def grad_mag_u8(img_u8, sigma=1.2):
    """Gradient magnitude — the illumination-robust representation used by the orientation
    search, the texture screen and the ECC refinement."""
    f = np.asarray(img_u8, dtype=np.float32)
    gx = cv2.Sobel(f, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(f, cv2.CV_32F, 0, 1, ksize=3)
    g = cv2.GaussianBlur(cv2.magnitude(gx, gy), (0, 0), sigma)
    return to_uint8(g, 1, 99)

def texture_score(img_u8):
    """Cheap 'is there anything here to match' number. Used to skip dead tiles before the
    GPU is asked to look at them — at 81 deg incidence a lot of tiles are pure shadow."""
    a = np.asarray(img_u8, np.float32)
    if a.size < 64:
        return 0.0
    return float(cv2.Laplacian(a, cv2.CV_32F, ksize=3).var())

print("Appearance preprocessing ready (no radiometric calibration in this notebook).")

In [ ]:
# 8b) Build the coarse variants and look at them
import matplotlib.pyplot as plt
from PIL import Image

ohrc_c_pc, nac_c_pc, ohrc_c_sg, nac_c_sg = prep_pair(ohrc_coarse, nac_coarse)

def to_rgb_pil(gray_u8):
    return Image.fromarray(np.stack([gray_u8] * 3, axis=-1))

fig, ax = plt.subplots(2, 2, figsize=(11, 9))
ax[0, 0].imshow(ohrc_c_pc, cmap='gray'); ax[0, 0].set_title(f'OHRC plain {ohrc_c_pc.shape}')
ax[0, 1].imshow(nac_c_pc,  cmap='gray'); ax[0, 1].set_title(f'NAC plain {nac_c_pc.shape}')
ax[1, 0].imshow(ohrc_c_sg, cmap='gray'); ax[1, 0].set_title(f'OHRC CLAHE (clip={OHRC_CLIP_LIMIT}) -> SuperGlue')
ax[1, 1].imshow(nac_c_sg,  cmap='gray'); ax[1, 1].set_title('NAC CLAHE + hist-matched -> SuperGlue')
for a in ax.ravel(): a.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'{PAIR_ID}_v4_01_preprocessed.png'), dpi=140)
plt.show()

print(f"texture score  OHRC {texture_score(ohrc_c_pc):9.1f}   NAC {texture_score(nac_c_pc):9.1f}")

In [ ]:
# 9) Orientation search + pre-flight pair gate  [v4: REPLACES v3's ORB flip test]
#
# v3 scored each flip with ORB + RANSAC inliers. ORB is an intensity-gradient descriptor
# run across two different sensors; it returned near-ties, so all four candidates were
# pushed into the expensive coarse stage.
#
# Dense ZNCC on gradient magnitude is better here and costs milliseconds:
#   - TM_CCOEFF_NORMED is invariant to any affine brightness change a*x + b, which is
#     exactly the freedom between two instruments.
#   - gradient magnitude discards absolute brightness and keeps structure.
#
# Two things this gets right that a naive version does not:
#   1. FIXED TEMPLATE FOOTPRINT. Sliding a small template over a large image gives more
#      trial positions and therefore a higher max correlation BY CHANCE. Left uncorrected
#      the search is biased toward small scales. Every scale is evaluated with a template
#      covering the same fraction of the search image, so the trial counts are comparable.
#   2. AN INDEPENDENT NULL. The noise floor cannot come from the other flips -- if the true
#      answer is "vertical", using vertical as the null puts the truth in the floor and the
#      margin collapses to 1.0. Instead the null is B with its blocks shuffled: identical
#      texture statistics, no spatial correspondence left.
NCC_SCALES_TIGHT = np.arange(0.80, 1.26, 0.02)   # both images already at a common GSD
NCC_SCALES_WIDE  = np.arange(0.50, 2.51, 0.02)   # raw products, scale unknown

def _zgrad(x):
    g = grad_mag_u8(x).astype(np.float32)
    return (g - g.mean()) / (g.std() + 1e-9)

def _phase_randomize(img, seed=0):
    """Surrogate with the SAME power spectrum but no spatial structure.

    This is the null hypothesis for the gate. It must be independent of which flip is
    true -- using the other flips as the null puts the truth in the floor whenever the
    truth happens to be one of them, and the margin collapses to 1.0.
    """
    f = np.asarray(img, np.float32)
    F = np.fft.fft2(f)
    r = np.random.default_rng(seed)
    ph = r.uniform(-np.pi, np.pi, F.shape)
    return to_uint8(np.real(np.fft.ifft2(np.abs(F) * np.exp(1j * ph))))

MIN_TEMPLATE_COVER = 0.35   # template area / search area

def _best_ncc(A_z, A_img, B, scales):
    """Best ZNCC of B (at any scale in `scales`) against A.

    Whichever image is smaller at a given scale becomes the TEMPLATE and the other is the
    search image -- no zero- or reflect-padding, so no invented terrain for the matcher to
    correlate against, and the roles stay symmetric as the scale sweeps past 1.

    Scales whose template covers less than MIN_TEMPLATE_COVER of the search image are
    skipped: a smaller template means more trial positions and therefore a higher max
    correlation by chance, so bounding coverage bounds that bias.
    """
    best = (-1.0, None, None)
    for s in scales:
        h, w = int(round(B.shape[0] * s)), int(round(B.shape[1] * s))
        if h < 24 or w < 16 or h > 4000 or w > 4000:
            continue
        Bs = cv2.resize(B, (w, h), interpolation=cv2.INTER_CUBIC)
        if h <= A_img.shape[0] and w <= A_img.shape[1]:
            search_z, tpl_z = A_z, _zgrad(Bs)
            search_shape, tpl_shape = A_img.shape, (h, w)
        elif A_img.shape[0] <= h and A_img.shape[1] <= w:
            search_z, tpl_z = _zgrad(Bs), A_z              # roles swapped
            search_shape, tpl_shape = (h, w), A_img.shape
        else:
            continue                                        # neither contains the other
        if (tpl_shape[0] * tpl_shape[1]) / float(search_shape[0] * search_shape[1]) < MIN_TEMPLATE_COVER:
            continue
        try:
            res = cv2.matchTemplate(search_z, tpl_z, cv2.TM_CCOEFF_NORMED)
        except cv2.error:
            continue
        _, mx, _, loc = cv2.minMaxLoc(res)
        if mx > best[0]:
            best = (float(mx), round(float(s), 3), loc)
    return best

def ncc_orientation_search(a_u8, b_u8, scales=None, max_side=400):
    """Returns (ranked_rows, floor). Rows are (ncc, flip_name, flip_code, scale, offset)."""
    scales = NCC_SCALES_TIGHT if scales is None else scales

    # ONE shared shrink factor for both images. Shrinking each to `max_side` independently
    # would normalise away the very scale ratio the search exists to measure -- the search
    # would then report 1.00 for every pair regardless of the truth.
    _k = min(1.0, max_side / max(max(a_u8.shape[:2]), max(b_u8.shape[:2])))
    def shrink(x):
        return cv2.resize(x, None, fx=_k, fy=_k, interpolation=cv2.INTER_AREA) if _k < 1.0 else x

    A, B = shrink(a_u8), shrink(b_u8)
    A_z = _zgrad(A)
    rows = []
    for name, code, Bv in (("no flip",    None, B),
                           ("horizontal",    1, B[:, ::-1].copy()),
                           ("vertical",      0, B[::-1, :].copy()),
                           ("both",         -1, B[::-1, ::-1].copy())):
        mx, sc, loc = _best_ncc(A_z, A, Bv, scales)
        if sc is not None:
            rows.append((mx, name, code, sc, loc))
    rows.sort(key=lambda t: -t[0])

    # Independent null: three shuffles of B, take the strongest spurious correlation.
    floor = max([_best_ncc(A_z, A, _phase_randomize(B, seed=k), scales)[0]
                 for k in range(3)] or [1e-6])
    return rows, max(float(floor), 1e-6)

# Calibrated against: 4 synthetic flips (x8.7-9.8), a synthetic 35-degree sun-angle change
# (x10.9), the real OHRC/NAC preview strips (x3.1), and three negatives -- unrelated
# synthetic terrain (x2.11), a phase-randomised surrogate (x1.16) and a self-shifted crop
# (x1.53). 2.5 sits in the gap, but the weakest TRUE positive is only x3.1, so this gate is
# ADVISORY: it never raises. A gate that halts a correct run is worse than no gate.
MIN_NCC_MARGIN = 2.5     # best / phase-randomised null
MIN_NCC_ABS    = 0.12    # absolute floor, so a near-zero null cannot manufacture a pass
STOP_ON_NO_OVERLAP = False   # set True to make a failed gate raise instead of warn

def summarise_orientation(rows, floor, verbose=True):
    if not rows:
        return dict(verdict="NO DATA", flip_candidates=[None], best=None, margin=0.0, floor=floor)
    top = rows[0]
    margin = top[0] / max(floor, 1e-6)
    runner_up = rows[1][0] if len(rows) > 1 else 0.0
    separation = top[0] / max(runner_up, 1e-6)

    if verbose:
        print(f'  {"orientation":<12}{"NCC":>8}  {"scale":>6}   offset(x,y)')
        for r in rows:
            print(f'  {r[1]:<12}{r[0]:8.3f}  {r[3]:6.2f}   {r[4]}')
        print(f'\n  shuffled-null floor : {floor:.3f}   (independent of which flip is true)')
        print(f'  best "{top[1]}"        : {top[0]:.3f}  -> margin over null x{margin:.1f}')
        print(f'  separation from runner-up ({rows[1][1] if len(rows)>1 else "-"}): x{separation:.2f}')

    if margin >= MIN_NCC_MARGIN and top[0] >= MIN_NCC_ABS:
        verdict = "SAME GROUND"
    elif margin >= 1.5:
        verdict = "INCONCLUSIVE"
    else:
        verdict = "NO DETECTABLE OVERLAP"

    # Commit to one flip only when it is both above the null AND clear of the runner-up.
    if verdict == "SAME GROUND" and separation >= 1.15:
        flip_candidates = [top[2]]
    else:
        flip_candidates = [r[2] for r in rows]

    if verbose:
        print(f'  VERDICT: {verdict}   flip candidates -> {flip_candidates}')
        if verdict == "NO DETECTABLE OVERLAP":
            print('  *** The dense check found nothing above its own noise floor. That usually\n'
                  '      means the two products image different ground, or the NAC corners are\n'
                  '      wrong. It can ALSO happen on a real pair with a large rotation, which\n'
                  '      this check does not search. Continuing anyway with all four flips --\n'
                  '      SuperGlue decides. Set STOP_ON_NO_OVERLAP = True to halt here. ***')
        elif verdict == "INCONCLUSIVE":
            print('  (passing all four flips to the coarse stage rather than guessing)')
    if verdict == "NO DETECTABLE OVERLAP" and STOP_ON_NO_OVERLAP:
        raise RuntimeError("Pre-flight gate found no overlap and STOP_ON_NO_OVERLAP is set.")
    return dict(verdict=verdict, flip_candidates=flip_candidates, best=top,
                margin=float(margin), separation=float(separation), floor=float(floor))

In [ ]:
# 9b) Run the pre-flight gate and pick orientation candidates
# Both coarse images are already at a common GSD, so the expected scale is 1.0 and the tight
# band is the right one. A scale far from 1.0 here means the GSD values are wrong.
_t0 = time.time()
ncc_rows, ncc_floor = ncc_orientation_search(ohrc_c_pc, nac_c_pc, scales=NCC_SCALES_TIGHT)
gate = summarise_orientation(ncc_rows, ncc_floor, verbose=True)
print(f"  ({time.time()-_t0:.1f}s)")

FLIP_CANDIDATES = gate["flip_candidates"]
if gate["best"] is not None and abs(gate["best"][3] - 1.0) > 0.15:
    print(f"\n*** The dense search prefers scale {gate['best'][3]:.2f}, not ~1.0, even though "
          f"both images were resampled to {TARGET_GSD_FINE_M:.3f} m/px.\n"
          f"    That points at a wrong GSD in one of the labels. Check the values in cell 7. ***")

In [ ]:
# 9c) Residual rotation — log-polar phase correlation, with a self-test
# The flip candidates cover NAC-L/R handedness. Two orbital passes can also leave an
# arbitrary in-between angle, and SuperPoint/SuperGlue are only weakly rotation-invariant.
#
# Kept from v3, all three fixes intact: square padding before the canvas resize (otherwise
# each strip gets a different anisotropic squash, manufacturing a rotation that is not
# there), both 180-degree twins carried (an FFT magnitude spectrum is centrosymmetric, so
# the angle is only recoverable mod 180), and BORDER_CONSTANT rather than BORDER_REFLECT.
def _pad_square(img):
    a = np.asarray(img, dtype=np.float32)
    h, w = a.shape[:2]
    n = max(h, w)
    out = np.zeros((n, n), np.float32)
    r, c = (n - h) // 2, (n - w) // 2
    out[r:r + h, c:c + w] = a
    return out

def estimate_rotation_deg(img_a, img_b, canvas=512):
    a = cv2.resize(_pad_square(img_a), (canvas, canvas), interpolation=cv2.INTER_AREA)
    b = cv2.resize(_pad_square(img_b), (canvas, canvas), interpolation=cv2.INTER_AREA)
    win = cv2.createHanningWindow((canvas, canvas), cv2.CV_32F)
    a, b = a * win, b * win

    def log_polar_mag(img):
        mag = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(img)))).astype(np.float32)
        return cv2.warpPolar(mag, (canvas, canvas), (canvas / 2.0, canvas / 2.0),
                             canvas / 2.0, cv2.WARP_POLAR_LOG + cv2.INTER_LINEAR)

    (_, shift_y), response = cv2.phaseCorrelate(log_polar_mag(a), log_polar_mag(b))
    return float(shift_y * (360.0 / canvas)), float(response)

_TEST_ANGLE = 7.0
_h, _w = ohrc_c_pc.shape
_rot = cv2.warpAffine(ohrc_c_pc, cv2.getRotationMatrix2D((_w / 2, _h / 2), _TEST_ANGLE, 1.0),
                      (_w, _h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
_recovered, _resp = estimate_rotation_deg(ohrc_c_pc, _rot)
ROT_SIGN = -1.0 if abs(_recovered - _TEST_ANGLE) <= abs(_recovered + _TEST_ANGLE) else 1.0
_err = abs(abs(_recovered) - _TEST_ANGLE)
ROTATION_ESTIMATOR_TRUSTED = _err < 2.0
print(f"Self-test: injected {_TEST_ANGLE:+.1f} deg, recovered {_recovered:+.2f} deg "
      f"(error {_err:.2f}, response {_resp:.3f}) -> ROT_SIGN={ROT_SIGN:+.0f}, "
      f"trusted={ROTATION_ESTIMATOR_TRUSTED}")

_nac_for_rot = nac_c_pc if FLIP_CANDIDATES[0] is None else cv2.flip(nac_c_pc, FLIP_CANDIDATES[0])
ROTATION_DEG, ROTATION_CONFIDENCE = estimate_rotation_deg(ohrc_c_pc, _nac_for_rot)
_applied = ROT_SIGN * ROTATION_DEG
print(f"Estimated residual rotation {ROTATION_DEG:+.2f} deg (response {ROTATION_CONFIDENCE:.3f})")

def _wrap180(a):
    return ((a + 180.0) % 360.0) - 180.0

if not ROTATION_ESTIMATOR_TRUSTED:
    ROT_CANDIDATES = [0.0]
elif abs(_applied) < 1.0:
    ROT_CANDIDATES = [0.0]
else:
    ROT_CANDIDATES = [0.0, _wrap180(_applied), _wrap180(_applied + 180.0)]

print(f"ROT_CANDIDATES (deg applied to NAC) = {[f'{c:+.2f}' for c in ROT_CANDIDATES]}")
print(f"Coarse stage will try {len(FLIP_CANDIDATES)} x {len(ROT_CANDIDATES)} = "
      f"{len(FLIP_CANDIDATES)*len(ROT_CANDIDATES)} orientation combinations.")

In [ ]:
# 10) SuperPoint + SuperGlue
# FIX B1 (from v3, kept): the checkpoint's preprocessor ships do_resize=True with
# size 480x640. Handing it two long thin strips squashes each by a DIFFERENT anisotropic
# factor and destroys the geometry before SuperPoint runs. Square padding + a square
# processor size fixes it; the padding offset is subtracted from the returned keypoints.
#
# v4: the input size is per call. v3 used one 1024 setting, which upsampled every 640 px
# tile by 1.6x for nothing.
from transformers import AutoImageProcessor, AutoModelForKeypointMatching
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT = "magic-leap-community/superglue_outdoor"    # MegaDepth: wide baseline
sg_processor = AutoImageProcessor.from_pretrained(CHECKPOINT)
sg_model = AutoModelForKeypointMatching.from_pretrained(CHECKPOINT).to(device).eval()

SG_SIZE_COARSE     = 1024
SG_SIZE_FINE       = 640     # == TILE, so tiles are not resampled at all
SG_MATCH_THRESHOLD = 0.15    # MAGSAC downstream absorbs a loose threshold better than a
                             # tight one recovers matches that were never emitted
print(f"device={device}  checkpoint default size={getattr(sg_processor,'size',None)} "
      f"(overridden per call to {SG_SIZE_COARSE} coarse / {SG_SIZE_FINE} fine)")

def pad_to_square(img_u8):
    h, w = img_u8.shape[:2]
    n = max(h, w)
    if n == h == w:
        return img_u8, 0, 0
    out = np.zeros((n, n), np.uint8)
    ox, oy = (n - w) // 2, (n - h) // 2
    out[oy:oy + h, ox:ox + w] = img_u8
    return out, ox, oy

_EMPTY = (np.zeros((0, 2), np.float32), np.zeros((0, 2), np.float32), np.zeros(0, np.float32))

def match_superglue(a_u8, b_u8, threshold=SG_MATCH_THRESHOLD, size=SG_SIZE_COARSE):
    """(kpts_a, kpts_b, scores) in the ORIGINAL coordinates of a_u8 / b_u8."""
    if a_u8.size == 0 or b_u8.size == 0:
        return _EMPTY
    a_sq, ax, ay = pad_to_square(a_u8)
    b_sq, bx, by = pad_to_square(b_u8)
    images = [to_rgb_pil(a_sq), to_rgb_pil(b_sq)]
    sg_processor.size = {"height": int(size), "width": int(size)}
    inputs = sg_processor(images, return_tensors="pt").to(device)
    with torch.inference_mode():
        raw = sg_model(**inputs)
    sizes = [[(im.height, im.width) for im in images]]
    out = sg_processor.post_process_keypoint_matching(raw, sizes, threshold=threshold)[0]
    k0 = out["keypoints0"].float().cpu().numpy().reshape(-1, 2) - np.array([ax, ay], np.float32)
    k1 = out["keypoints1"].float().cpu().numpy().reshape(-1, 2) - np.array([bx, by], np.float32)
    sc = out["matching_scores"].float().cpu().numpy().reshape(-1)
    keep = ((k0[:, 0] >= 0) & (k0[:, 1] >= 0) &
            (k0[:, 0] < a_u8.shape[1]) & (k0[:, 1] < a_u8.shape[0]) &
            (k1[:, 0] >= 0) & (k1[:, 1] >= 0) &
            (k1[:, 0] < b_u8.shape[1]) & (k1[:, 1] < b_u8.shape[0]))
    return k0[keep], k1[keep], sc[keep]

_t0 = time.time()
_k0, _k1, _sc = match_superglue(ohrc_c_sg, nac_c_sg)
print(f"Smoke test on the coarse pair: {len(_k0)} matches at threshold {SG_MATCH_THRESHOLD} "
      f"in {time.time()-_t0:.1f}s"
      + (f" (scores {_sc.min():.2f}-{_sc.max():.2f})" if len(_sc) else ""))
if len(_k0) == 0:
    print("*** No coarse matches. Check the preprocessed preview above before continuing. ***")

In [ ]:
# 10c) Dense tile matcher — no neural network involved, so it keeps working where
#      SuperGlue does not. Lives in its own cell because it needs only cv2/numpy.
# Dense tile matcher — the fallback for when SuperGlue will not match this imagery.
#
# On a low-sun pair the FINE-scale appearance is dominated by shadows that differ between
# the two acquisitions, so SuperPoint fires on shadow edges that have no counterpart and
# SuperGlue returns confident nonsense. Dense correlation on gradient magnitude does not
# care: it integrates over the whole patch instead of betting on individual keypoints, and
# the coarse band check on this pair proves it works where the feature matcher does not.
#
# It returns ONE sub-pixel displacement per patch instead of many keypoints, so it trades
# density for reliability — with a few hundred tiles that is still plenty of tie points,
# and every one carries an NCC margin against a phase-randomised null.
DENSE_TILE_MIN_MARGIN = 2.0
DENSE_TILE_SUBDIV     = 3       # NxN sub-patches per tile. 3 gives ~9 tie points
                                # per tile instead of 4 at no extra model cost,
                                # and the consensus step needs repeats to average.

def _zg(x):
    g = grad_mag_u8(x).astype(np.float32)
    return (g - g.mean()) / (g.std() + 1e-9)

def match_dense(warped_u8, ref_u8, valid_mask, subdiv=DENSE_TILE_SUBDIV,
                min_margin=DENSE_TILE_MIN_MARGIN):
    """(kpts_a, kpts_b, scores) — one correspondence per sub-patch that clears the null."""
    h, w = ref_u8.shape[:2]
    ys, xs = np.where(valid_mask)
    if len(ys) < 400:
        return _EMPTY
    Y0, Y1, X0, X1 = int(ys.min()), int(ys.max()), int(xs.min()), int(xs.max())
    b_z = _zg(ref_u8)
    a_z = _zg(warped_u8)
    ka, kb, sc = [], [], []
    for iy in range(subdiv):
        for ix in range(subdiv):
            ph = (Y1 - Y0) // subdiv; pw = (X1 - X0) // subdiv
            if ph < 64 or pw < 64:
                continue
            ty = Y0 + iy * ph; tx = X0 + ix * pw
            th = int(ph * 0.85); tw_ = int(pw * 0.85)
            ty += (ph - th) // 2; tx += (pw - tw_) // 2
            if valid_mask[ty:ty + th, tx:tx + tw_].mean() < 0.9:
                continue
            tpl = a_z[ty:ty + th, tx:tx + tw_]
            if tpl.shape[0] > b_z.shape[0] or tpl.shape[1] > b_z.shape[1]:
                continue
            try:
                _, mx, _, loc = cv2.minMaxLoc(cv2.matchTemplate(b_z, tpl, cv2.TM_CCOEFF_NORMED))
                null = max(cv2.minMaxLoc(cv2.matchTemplate(
                    _zg(_phase_randomize(ref_u8, seed=q)), tpl, cv2.TM_CCOEFF_NORMED))[1]
                    for q in range(2))
            except cv2.error:
                continue
            if mx / max(null, 1e-6) < min_margin:
                continue
            dx, dy = loc[0] - tx, loc[1] - ty
            # sub-pixel: phase-correlate the now-aligned patches
            sy0, sx0 = int(ty + dy), int(tx + dx)
            if sy0 < 0 or sx0 < 0 or sy0 + th > h or sx0 + tw_ > w:
                continue
            A = a_z[ty:ty + th, tx:tx + tw_]
            B = b_z[sy0:sy0 + th, sx0:sx0 + tw_]
            try:
                win = cv2.createHanningWindow((tw_, th), cv2.CV_32F)
                (sdx, sdy), _r = cv2.phaseCorrelate(np.ascontiguousarray(A) * win,
                                                    np.ascontiguousarray(B) * win)
                if abs(sdx) > 2 or abs(sdy) > 2:
                    sdx = sdy = 0.0
            except cv2.error:
                sdx = sdy = 0.0
            cx, cy = tx + tw_ / 2.0, ty + th / 2.0
            ka.append([cx, cy]); kb.append([cx + dx + sdx, cy + dy + sdy])
            sc.append(float(min(1.0, mx)))
    if not ka:
        return _EMPTY
    return (np.asarray(ka, np.float32), np.asarray(kb, np.float32), np.asarray(sc, np.float32))

print(f"Dense tile matcher ready ({DENSE_TILE_SUBDIV}x{DENSE_TILE_SUBDIV} sub-patches, "
      f"margin >= {DENSE_TILE_MIN_MARGIN}).")


In [ ]:
# 10b) RIFT2 — OFF by default
# RIFT2 describes keypoints from phase congruency, so its illumination invariance is
# structural rather than inherited from a training distribution. That makes it a genuine
# hedge on the sun-angle problem the problem statement names.
#
# It is off because it is the slow part (phase congruency is CPU-bound, and it runs per
# tile), and because nobody has yet measured whether SuperGlue needs the help now that the
# B1 preprocessing bug is fixed — the comparison that made RIFT2 look necessary was run
# against a SuperGlue that was being fed anisotropically squashed strips.
#
# Set USE_RIFT2 = True in the config cell to run the ablation. The per-matcher inlier
# breakdown is reported either way, so one run answers the question.
if USE_RIFT2:
    import sys, subprocess
    RIFT2_DIR = "/content/RIFT2-multimodal-matching-rotation-python"
    if not os.path.exists(RIFT2_DIR):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/canyagmur/RIFT2-multimodal-matching-rotation-python.git",
                        RIFT2_DIR], check=True)
        subprocess.run(["pip", "install", "-q", "-r",
                        os.path.join(RIFT2_DIR, "requirements.txt")], check=False)
    if RIFT2_DIR not in sys.path:
        sys.path.insert(0, RIFT2_DIR)
    from src.RIFT2 import RIFT2
    from src.matcher_functions import match_keypoints_nn, outlier_removal
    rift2_pipeline = RIFT2()

    RIFT2_RATIO, RIFT2_MUTUAL, RIFT2_MAX_SIDE = 0.85, True, 1200

    def match_rift2(a_u8, b_u8):
        sa = min(1.0, RIFT2_MAX_SIDE / max(a_u8.shape[:2]))
        sb = min(1.0, RIFT2_MAX_SIDE / max(b_u8.shape[:2]))
        a = cv2.resize(a_u8, None, fx=sa, fy=sa, interpolation=cv2.INTER_AREA) if sa < 1 else a_u8
        b = cv2.resize(b_u8, None, fx=sb, fy=sb, interpolation=cv2.INTER_AREA) if sb < 1 else b_u8
        kp_a, des_a, kp_b, des_b = rift2_pipeline(cv2.cvtColor(a, cv2.COLOR_GRAY2BGR),
                                                  cv2.cvtColor(b, cv2.COLOR_GRAY2BGR))
        if des_a is None or des_b is None or len(kp_a) < 4 or len(kp_b) < 4:
            return _EMPTY
        pa_, pb_, matches = match_keypoints_nn(des_a, des_b, kp_a, kp_b,
                                               lowes_ratio=RIFT2_RATIO, mutual=RIFT2_MUTUAL)
        pa = np.asarray(pa_, np.float32).reshape(-1, 2)
        pb = np.asarray(pb_, np.float32).reshape(-1, 2)
        try:
            d = np.asarray([m.distance for m in matches], np.float32)
            rng_ = float(d.max() - d.min())
            sc = (1.0 - (d - d.min()) / (rng_ + 1e-9)) if rng_ > 0 else np.ones(len(pa), np.float32)
            sc = sc[:len(pa)]
        except Exception:
            sc = np.ones(len(pa), np.float32)
        if len(sc) != len(pa):
            sc = np.ones(len(pa), np.float32)
        if len(pa) >= 4:
            try:
                res = outlier_removal(pa, pb)
                pa_o = np.asarray(res[0], np.float32).reshape(-1, 2)
                pb_o = np.asarray(res[1], np.float32).reshape(-1, 2)
                if 4 <= len(pa_o) <= len(pa):
                    pa, pb, sc = pa_o, pb_o, np.ones(len(pa_o), np.float32)
            except Exception as e:
                print("  (outlier_removal skipped:", type(e).__name__, ")")
        return pa / sa, pb / sb, sc

    _t0 = time.time()
    _r0, _r1, _rs = match_rift2(ohrc_c_pc, nac_c_pc)
    print(f"RIFT2 smoke test: {len(_r0)} matches in {time.time()-_t0:.1f}s")
else:
    def match_rift2(a_u8, b_u8):
        return _EMPTY
    print("RIFT2 disabled (USE_RIFT2 = False). SuperGlue only — no clone, no CPU cost.")

In [ ]:
# 11) Staged, guarded model fitting
# A homography from 4 points has zero residual by construction — which is why v1 reported
# rmse_px exactly 0.0 next to a visibly broken warp — and is wrong everywhere else. Two
# line-scan strips over relief are not related by a single homography anyway.
#
# So: fit the simplest model the evidence supports, and upgrade only on inlier count AND
# spatial coverage.
#
# FIX C1: `min_minor_axis_px` is now an actual parameter. v3.2's cell 12 passed it to a
# function that did not accept it -> TypeError, every run, right after the fine stage.
RANSAC_THRESH_PX        = 3.0
AFFINE_MIN_INLIERS      = 20
HOMOGRAPHY_MIN_INLIERS  = 40
HOMOGRAPHY_MIN_COVERAGE = 0.10
MIN_COVERAGE_TO_TRUST   = 0.05
MIN_INLIERS_TO_TRUST    = 15
DEFAULT_MIN_MINOR_AXIS_PX = 5.0   # perpendicular spread below this is agenuine line

_USAC = getattr(cv2, 'USAC_MAGSAC', cv2.RANSAC)

def hull_coverage(pts, frame_shape):
    pts = np.asarray(pts, np.float32).reshape(-1, 2)
    if len(pts) < 3:
        return 0.0
    return float(cv2.contourArea(cv2.convexHull(pts)) / (frame_shape[0] * frame_shape[1]))

def is_degenerate(pts, min_minor_axis_px=DEFAULT_MIN_MINOR_AXIS_PX):
    """True only when the points really do lie on a line.

    v4.1: this used to compare the eigenvalue RATIO of the point covariance against a
    threshold. That is the wrong test for an elongated footprint. A ratio is small whenever
    the MAJOR axis is large, however fat the minor axis is -- on this pair's 15000 x 2500 px
    overlap the ratio test declared "collinear" any cloud narrower than ~620 px, and duly
    rejected a correct fit whose convex hull covered 56% of the frame.

    The honest quantity is the minor axis in ABSOLUTE pixels: the standard deviation of the
    points perpendicular to their principal direction. Genuinely collinear points have a
    minor axis of order a pixel. Conditioning of the fit is a separate question, and hull
    coverage already answers it.
    """
    pts = np.asarray(pts, np.float64).reshape(-1, 2)
    if len(pts) < 3:
        return True
    ev = np.linalg.eigvalsh(np.cov((pts - pts.mean(0)).T) + 1e-12 * np.eye(2))
    return bool(np.sqrt(max(ev.min(), 0.0)) < min_minor_axis_px)

def minor_axis_px(pts):
    pts = np.asarray(pts, np.float64).reshape(-1, 2)
    if len(pts) < 3:
        return 0.0
    ev = np.linalg.eigvalsh(np.cov((pts - pts.mean(0)).T) + 1e-12 * np.eye(2))
    return float(np.sqrt(max(ev.min(), 0.0)))

def robust_fit(src, dst, frame_shape, thresh=RANSAC_THRESH_PX, verbose=True,
               min_minor_axis_px=DEFAULT_MIN_MINOR_AXIS_PX):
    """(H 3x3, inlier_mask, model_name) or (None, None, None)."""
    src = np.asarray(src, np.float32).reshape(-1, 1, 2)
    dst = np.asarray(dst, np.float32).reshape(-1, 1, 2)
    if len(src) < 4:
        return None, None, None
    best = (None, None, None)

    M, m = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC,
                                       ransacReprojThreshold=thresh,
                                       maxIters=20000, confidence=0.999)
    if M is not None and m is not None:
        best = (to_h(M), m.ravel().astype(bool), 'similarity')

    if best[1] is not None and best[1].sum() >= AFFINE_MIN_INLIERS:
        M, m = cv2.estimateAffine2D(src, dst, method=cv2.RANSAC,
                                    ransacReprojThreshold=thresh,
                                    maxIters=20000, confidence=0.999)
        if M is not None and m is not None and m.ravel().sum() >= best[1].sum():
            best = (to_h(M), m.ravel().astype(bool), 'affine')

    if best[1] is not None and best[1].sum() >= HOMOGRAPHY_MIN_INLIERS:
        cov = hull_coverage(dst.reshape(-1, 2)[best[1]], frame_shape)
        if cov >= HOMOGRAPHY_MIN_COVERAGE:
            H, m = cv2.findHomography(src, dst, _USAC, thresh,
                                      maxIters=50000, confidence=0.9995)
            if H is not None and m is not None and m.ravel().sum() >= best[1].sum():
                best = (H, m.ravel().astype(bool), 'homography')
        elif verbose:
            print(f'  homography withheld: coverage {cov:.1%} < {HOMOGRAPHY_MIN_COVERAGE:.0%}')

    H, mask, name = best
    if H is None:
        return None, None, None
    if is_degenerate(dst.reshape(-1, 2)[mask], min_minor_axis_px=min_minor_axis_px):
        if verbose:
            print(f'  rejected: inliers effectively collinear (axis ratio < {min_minor_axis_px})')
        return None, None, None
    return H, mask, name

print(f"similarity always; affine at >={AFFINE_MIN_INLIERS} inliers; homography at "
      f">={HOMOGRAPHY_MIN_INLIERS} inliers and >={HOMOGRAPHY_MIN_COVERAGE:.0%} coverage.")
print("RANSAC backend:", 'USAC_MAGSAC' if _USAC != cv2.RANSAC else 'RANSAC (MAGSAC unavailable)')

In [ ]:
# 12) COARSE stage — GEOMETRIC PRIOR first, matching only to refine it
#
# The previous version searched flip x rotation blindly and let SuperGlue vote. That was
# always the wrong shape for this problem: BOTH images are georeferenced. OHRC's corners
# come from its PDS4 label and NAC's from KNOWN_NAC_CORNERS, so composing the two corner
# models gives the OHRC->NAC transform ANALYTICALLY -- mirror, rotation, scale and shift all
# included, exact at the corners. On the real pair this reproduces the NAC pixel coordinates
# of all four OHRC corners to ~1e-13 px.
#
# The blind search was not only unnecessary, it was harmful: the log-polar estimator's 180
# degree twin won on 12 inliers out of 1208 (1%), and the resulting similarity matrix was
# singular, which is what crashed the tiled stage.
#
# So: compute the prior, then use SuperGlue only to CORRECT it. Matching a pre-warped,
# nearly-aligned pair is the regime SuperGlue was trained for, and a correction that fails
# to beat the prior is simply discarded.

def latlon_affine(corners_deg, shape, lon_ref):
    """[lat, lon]^T = M @ [row, col]^T + t   for one image."""
    n_l, n_s = shape
    keys = ("UL", "UR", "LL", "LR")
    pix = np.array([[0, 0], [0, n_s - 1], [n_l - 1, 0], [n_l - 1, n_s - 1]], float)
    lat = np.array([corners_deg[k]["lat"] for k in keys], float)
    lon = np.array([float(unwrap_lon(corners_deg[k]["lon"], lon_ref)) for k in keys])
    A = np.column_stack([pix[:, 0], pix[:, 1], np.ones(4)])
    cl, *_ = np.linalg.lstsq(A, lat, rcond=None)
    cn, *_ = np.linalg.lstsq(A, lon, rcond=None)
    M = np.array([cl[:2], cn[:2]]); t = np.array([cl[2], cn[2]])
    if abs(np.linalg.det(M)) < 1e-18:
        raise ValueError("Degenerate corner model — the four corners do not span a patch.")
    return M, t

_Mo, _to = latlon_affine(ohrc_corners, ohrc_shape, LON_REF)
_Mn, _tn = latlon_affine(nac_corners,  nac_shape,  LON_REF)
_R = np.linalg.inv(_Mn) @ _Mo                      # (row,col)_ohrc -> (row,col)_nac
_T = np.linalg.inv(_Mn) @ (_to - _tn)

# into the notebook's (x=col, y=row) convention
H_native_geo = np.array([[_R[1, 1], _R[1, 0], _T[1]],
                         [_R[0, 1], _R[0, 0], _T[0]],
                         [0.0, 0.0, 1.0]])
H_fine_geo = M_nac_native_to_fine @ H_native_geo @ np.linalg.inv(M_ohrc_native_to_fine)

_A = H_native_geo[:2, :2]
_det = float(np.linalg.det(_A))
_sx, _sy = float(np.hypot(*_A[:, 0])), float(np.hypot(*_A[:, 1]))
print("GEOMETRIC PRIOR from the two corner models (no matching involved):")
print(f"  scale x {_sx:.4f}, y {_sy:.4f}   (GSD ratio OHRC/NAC = "
      f"{OHRC_NATIVE_GSD_M/NAC_NATIVE_GSD_M:.4f})")
print(f"  determinant {_det:+.4f} -> the pair is "
      f"{'MIRRORED (opposite column sense)' if _det < 0 else 'not mirrored'}")
_res = max(np.linalg.norm(apply_h(H_native_geo, [[c, r]])[0] -
                          np.linalg.solve(_Mn, np.array([v['lat'],
                              float(unwrap_lon(v['lon'], LON_REF))]) - _tn)[::-1])
           for (r, c), v in [((0, 0), ohrc_corners['UL']),
                             ((0, ohrc_shape[1]-1), ohrc_corners['UR']),
                             ((ohrc_shape[0]-1, 0), ohrc_corners['LL']),
                             ((ohrc_shape[0]-1, ohrc_shape[1]-1), ohrc_corners['LR'])])
print(f"  corner round-trip residual: {_res:.1f} native NAC px")
print( "  (this is NOT expected to be ~0. A 6-DOF affine fitted through 4 corners is exact")
print( "   only when they form a parallelogram, and a real ground track does not. Tens of")
print( "   pixels here is normal and is the corner model's intrinsic error; hundreds means")
print( "   the two corner models genuinely disagree.)")

# Orientation is now a fact, not a search. These stay defined for the metrics below.
BEST_FLIP_CODE, BEST_ROT_DEG = None, 0.0
M_var_best = np.eye(3)
nac_c_sg_var, nac_c_pc_var = nac_c_sg, nac_c_pc

# --------------------------------------------------------------------------------------
# Refinement: pre-warp OHRC-coarse into the NAC-coarse frame with the prior, then match.
# Anything SuperGlue finds is a CORRECTION on top of the prior, not a transform from scratch.
# --------------------------------------------------------------------------------------
COARSE_BLOCK_ASPECT  = 1.5
COARSE_MAX_BLOCKS    = 6
COARSE_BLOCK_OVERLAP = 0.25
REFINE_MIN_INLIERS   = 25       # below this the correction is not trusted
REFINE_MAX_SHIFT_PX  = 250      # a "correction" larger than this is a mis-match, not a fix

def _blocks(img, max_blocks=COARSE_MAX_BLOCKS, aspect=COARSE_BLOCK_ASPECT,
            overlap=COARSE_BLOCK_OVERLAP):
    """Near-square tiles along the long axis. Square-padding a 7:1 strip to feed a square
    processor wastes most of the canvas; blocks keep the terrain at full resolution."""
    h, w = img.shape[:2]
    long_, short_ = (h, w) if h >= w else (w, h)
    if long_ <= aspect * max(short_, 1):
        return [(img, 0, 0)]
    n = int(min(max_blocks, max(2, round(long_ / float(short_)))))
    side = max(int(np.ceil(long_ / (1 + (n - 1) * (1 - overlap)))), short_)
    step = max(1, int(side * (1 - overlap)))
    starts, s = [], 0
    while len(starts) < n and s < long_ - side:
        starts.append(s); s += step
    starts.append(max(0, long_ - side))
    out = []
    for st in dict.fromkeys(starts):
        out.append((img[st:st + side, :], 0, st) if h >= w else (img[:, st:st + side], st, 0))
    return out

def match_blocked(a_u8, b_u8, size, threshold):
    A, B = _blocks(a_u8), _blocks(b_u8)
    ka_all, kb_all, sc_all = [], [], []
    for a_sub, axo, ayo in A:
        for b_sub, bxo, byo in B:
            if min(a_sub.shape[:2]) < 32 or min(b_sub.shape[:2]) < 32:
                continue
            ka, kb, sc = match_superglue(a_sub, b_sub, threshold=threshold, size=size)
            if len(ka) == 0:
                continue
            ka_all.append(ka + np.array([axo, ayo], np.float32))
            kb_all.append(kb + np.array([bxo, byo], np.float32))
            sc_all.append(sc)
    if not ka_all:
        return _EMPTY
    return (np.vstack(ka_all).astype(np.float32), np.vstack(kb_all).astype(np.float32),
            np.concatenate(sc_all).astype(np.float32))

H_coarse_geo = H_at_level(H_fine_geo, TARGET_GSD_FINE_M, TARGET_GSD_COARSE_M)
_h_n, _w_n = nac_c_sg.shape
ohrc_c_sg_warped = cv2.warpPerspective(ohrc_c_sg, H_coarse_geo.astype(np.float64),
                                       (_w_n, _h_n), flags=cv2.INTER_LINEAR, borderValue=0)
_valid_c = cv2.warpPerspective(np.full(ohrc_c_sg.shape, 255, np.uint8),
                               H_coarse_geo.astype(np.float64), (_w_n, _h_n),
                               flags=cv2.INTER_NEAREST, borderValue=0)
print(f"\nOHRC warped into the NAC coarse frame by the prior: "
      f"{100.0*(_valid_c > 0).mean():.1f}% of the NAC frame covered")

print(f"coarse blocking: warped OHRC {ohrc_c_sg_warped.shape} -> "
      f"{len(_blocks(ohrc_c_sg_warped))} blocks, NAC {nac_c_sg.shape} -> "
      f"{len(_blocks(nac_c_sg))} blocks")

# ---- Dense translation check, BEFORE feature matching -------------------------------
# The prior is affine through 4 corners; its dominant error is a bulk shift. Measuring that
# densely is far more robust than asking a feature matcher to discover it, and it costs
# milliseconds. Only then is SuperGlue asked for anything.
DENSE_MIN_MARGIN = 2.0

def _z(x):
    x = np.asarray(x, np.float32)
    return (x - x.mean()) / (x.std() + 1e-9)

def measure_prior_shift(warped_u8, ref_u8, frac_h=0.5, frac_w=0.6):
    a, b = _z(grad_mag_u8(warped_u8)), _z(grad_mag_u8(ref_u8))
    h, w = a.shape
    th, tw = max(32, int(h * frac_h)), max(32, int(w * frac_w))
    y0, x0 = (h - th) // 2, (w - tw) // 2
    tpl = a[y0:y0 + th, x0:x0 + tw]
    if tpl.shape[0] > b.shape[0] or tpl.shape[1] > b.shape[1]:
        return None
    _, mx, _, loc = cv2.minMaxLoc(cv2.matchTemplate(b, tpl, cv2.TM_CCOEFF_NORMED))
    null = max(cv2.minMaxLoc(cv2.matchTemplate(
        _z(_phase_randomize(ref_u8, seed=k)), tpl, cv2.TM_CCOEFF_NORMED))[1] for k in range(3))
    return dict(dx=loc[0] - x0, dy=loc[1] - y0, ncc=float(mx), null=float(null),
                margin=float(mx / max(null, 1e-6)))

# ---- Measure the prior's residual PER BAND, not once for the whole strip ----------------
# A single global shift can only remove a constant bias. If the prior DRIFTS along the strip
# -- which it does, because a 6-DOF affine through 4 corners cannot represent real pushbroom
# geometry with roll -6.2 deg and pitch +17.4 deg -- then one shift leaves a residual as
# large as the drift itself, and tiles far from where it was measured still get a bad window.
# Measuring band by band and interpolating turns a ~150 px leftover into ~20-30 px.
DENSE_BANDS      = 8
DENSE_BAND_SLACK = 160      # coarse px of vertical search room per band
DENSE_MIN_BANDS  = 3        # below this, fall back to a single global shift

def measure_band_shifts(warped_u8, ref_u8, valid_u8, n_bands=DENSE_BANDS,
                        slack=DENSE_BAND_SLACK):
    h, w = ref_u8.shape
    a_all, b_all = _z(grad_mag_u8(warped_u8)), _z(grad_mag_u8(ref_u8))
    edges = np.linspace(0, h, n_bands + 1).astype(int)
    rows = []
    for k in range(n_bands):
        y0, y1 = edges[k], edges[k + 1]
        if y1 - y0 < 64:
            continue
        cov = (valid_u8[y0:y1] > 0).mean()
        if cov < 0.35:                      # not enough warped OHRC in this band to match
            continue
        tw = max(32, int(w * 0.6)); x0 = (w - tw) // 2
        th = max(32, int((y1 - y0) * 0.8)); ty = y0 + ((y1 - y0) - th) // 2
        tpl = a_all[ty:ty + th, x0:x0 + tw]
        sy0, sy1 = max(0, ty - slack), min(h, ty + th + slack)
        seg = b_all[sy0:sy1]
        if tpl.shape[0] > seg.shape[0] or tpl.shape[1] > seg.shape[1]:
            continue
        _, mx, _, loc = cv2.minMaxLoc(cv2.matchTemplate(seg, tpl, cv2.TM_CCOEFF_NORMED))
        null = max(cv2.minMaxLoc(cv2.matchTemplate(
            _z(_phase_randomize(ref_u8[sy0:sy1], seed=q)), tpl,
            cv2.TM_CCOEFF_NORMED))[1] for q in range(2))
        rows.append(dict(yc=ty + th / 2.0, dx=loc[0] - x0, dy=(sy0 + loc[1]) - ty,
                         ncc=float(mx), margin=float(mx / max(null, 1e-6)), cov=float(cov)))
    return rows

_bands = measure_band_shifts(ohrc_c_sg_warped, nac_c_sg, _valid_c)
_good = [r for r in _bands if r['margin'] >= DENSE_MIN_MARGIN]
print(f"\nDense check, {len(_bands)} bands measured, {len(_good)} above margin "
      f"{DENSE_MIN_MARGIN}:")
for r in _bands:
    print(f"   y~{r['yc']:7.0f}  dx {r['dx']:+5d}  dy {r['dy']:+5d} coarse px   "
          f"NCC {r['ncc']:.3f}  margin x{r['margin']:4.1f}  cover {r['cov']:.0%}"
          f"{'' if r['margin'] >= DENSE_MIN_MARGIN else '   <- rejected'}")

PRIOR_SHIFT_COEF = None       # (coef_dx, coef_dy) in COARSE px vs coarse row
DENSE_APPLIED = False
if len(_good) >= DENSE_MIN_BANDS:
    _yc = np.array([r['yc'] for r in _good], float)
    _dx = np.array([r['dx'] for r in _good], float)
    _dy = np.array([r['dy'] for r in _good], float)
    _deg = 2 if len(_good) >= 5 else 1
    PRIOR_SHIFT_COEF = (np.polyfit(_yc, _dx, _deg), np.polyfit(_yc, _dy, _deg))
    PRIOR_SHIFT_RANGE = (float(_yc.min()), float(_yc.max()))
    _rx = _dx - np.polyval(PRIOR_SHIFT_COEF[0], _yc)
    _ry = _dy - np.polyval(PRIOR_SHIFT_COEF[1], _yc)
    DENSE_APPLIED = True
    print(f"  fitted a degree-{_deg} shift field over {len(_good)} bands; "
          f"band residual {np.abs(np.r_[_rx,_ry]).max():.1f} coarse px")
    print(f"  drift across the strip: dx {_dx.min():+.0f}..{_dx.max():+.0f}, "
          f"dy {_dy.min():+.0f}..{_dy.max():+.0f} coarse px "
          f"-- a single global shift would leave up to "
          f"{max(np.ptp(_dx), np.ptp(_dy))/2:.0f} px of that behind")
else:
    _ms_global = measure_prior_shift(ohrc_c_sg_warped, nac_c_sg)
    if _ms_global is not None and _ms_global['margin'] >= DENSE_MIN_MARGIN:
        PRIOR_SHIFT_COEF = (np.array([float(_ms_global['dx'])]),
                            np.array([float(_ms_global['dy'])]))
        DENSE_APPLIED = True
        print(f"  too few good bands — falling back to ONE global shift "
              f"({_ms_global['dx']:+d}, {_ms_global['dy']:+d}) coarse px, "
              f"margin x{_ms_global['margin']:.1f}")
    else:
        print(f"  no band and no global shift cleared margin {DENSE_MIN_MARGIN}. The two "
              f"coarse\n  images do not share detectable structure — look at the "
              f"preprocessed preview before\n  spending GPU time below.")

PRIOR_SHIFT_RANGE = None      # (y_min, y_max) of the bands actually fitted
PRIOR_SHIFT_BASE  = (0.0, 0.0)  # the shift already baked into H_fine0, in COARSE px

def _shift_at(yc):
    """Raw fitted shift at a coarse row, CLAMPED to the fitted range.

    A degree-2 polynomial extrapolates wildly. Band 1 is often rejected for low margin, so
    evaluating at y=0 is outside the data: on this pair that produced a spurious +48 fine px
    at the strip start. Outside the fitted span we hold the end value instead of inventing one.
    """
    if PRIOR_SHIFT_COEF is None:
        return 0.0, 0.0
    if PRIOR_SHIFT_RANGE is not None:
        yc = float(np.clip(yc, PRIOR_SHIFT_RANGE[0], PRIOR_SHIFT_RANGE[1]))
    return (float(np.polyval(PRIOR_SHIFT_COEF[0], yc)),
            float(np.polyval(PRIOR_SHIFT_COEF[1], yc)))

def prior_shift_fine(y_fine):
    """Correction RELATIVE to what H_fine0 already contains, in FINE px.

    Cell 12 bakes the mid-strip shift into H_coarse_geo (and thus into H_fine0). If the tiled
    stage then added the absolute shift as well, the same correction would be applied TWICE.
    Returning the residual makes the two stages compose instead of compound.
    """
    if PRIOR_SHIFT_COEF is None:
        return 0.0, 0.0
    k = TARGET_GSD_FINE_M / TARGET_GSD_COARSE_M
    dx, dy = _shift_at(float(y_fine) * k)
    return (dx - PRIOR_SHIFT_BASE[0]) / k, (dy - PRIOR_SHIFT_BASE[1]) / k

if DENSE_APPLIED:
    print(f"  baked into the prior: ({PRIOR_SHIFT_BASE[0]:+.1f}, {PRIOR_SHIFT_BASE[1]:+.1f}) "
          f"coarse px")
    _c0, _c1 = prior_shift_fine(0.0), prior_shift_fine(NAC_FINE_SHAPE[0])
    print(f"  RESIDUAL field the tiled stage adds on top, in FINE px: "
          f"start ({_c0[0]:+.1f}, {_c0[1]:+.1f}) -> end ({_c1[0]:+.1f}, {_c1[1]:+.1f})")
    print(f"  (near zero means the drift is a constant bias, already fully absorbed)")
    # re-warp the coarse preview with the mid-strip shift, for the block matching below
    _kc = TARGET_GSD_FINE_M / TARGET_GSD_COARSE_M
    _midc = _shift_at(NAC_FINE_SHAPE[0] * _kc / 2.0)
    PRIOR_SHIFT_BASE = _midc                      # record what we are baking in
    H_coarse_geo = mat_translate(_midc[0], _midc[1]) @ H_coarse_geo
    H_fine_geo   = H_at_level(H_coarse_geo, TARGET_GSD_COARSE_M, TARGET_GSD_FINE_M)
    ohrc_c_sg_warped = cv2.warpPerspective(ohrc_c_sg, H_coarse_geo.astype(np.float64),
                                           (_w_n, _h_n), flags=cv2.INTER_LINEAR, borderValue=0)
    _valid_c = cv2.warpPerspective(np.full(ohrc_c_sg.shape, 255, np.uint8),
                                   H_coarse_geo.astype(np.float64), (_w_n, _h_n),
                                   flags=cv2.INTER_NEAREST, borderValue=0)

sg0, sg1, sgs = match_blocked(ohrc_c_sg_warped, nac_c_sg, SG_SIZE_COARSE, SG_MATCH_THRESHOLD)
rf0, rf1, rfs = match_rift2(ohrc_c_sg_warped, nac_c_pc) if USE_RIFT2 else _EMPTY
print(f"Refinement matches: SuperGlue {len(sg0)}" + (f" + RIFT2 {len(rf0)}" if USE_RIFT2 else ""))

pool0 = np.vstack([sg0, rf0]) if len(rf0) else sg0
pool1 = np.vstack([sg1, rf1]) if len(rf0) else sg1
pools = np.concatenate([sgs, rfs]) if len(rf0) else sgs
pool_src = np.array(['superglue'] * len(sg0) + ['rift2'] * len(rf0))

H_fine0 = H_fine_geo
coarse_model = ('geometric prior + banded dense shift field' if DENSE_APPLIED
                else 'geometric prior')
coarse_mask, n_sg_in, n_rift2_in = None, 0, 0
REFINED = False

if len(pool0) >= 4:
    # correction lives in the WARPED frame: warped-ohrc -> nac, ideally identity
    C, coarse_mask, _m = robust_fit(pool0, pool1, nac_c_sg.shape, verbose=False)
    if C is not None:
        n_in = int(coarse_mask.sum())
        shift = float(np.linalg.norm(apply_h(C, [[_w_n/2, _h_n/2]])[0] - [_w_n/2, _h_n/2]))
        print(f"  correction: {n_in} inliers of {len(pool0)}, centre moves {shift:.1f} "
              f"coarse px ({shift*TARGET_GSD_COARSE_M:.0f} m)")
        if n_in >= REFINE_MIN_INLIERS and shift <= REFINE_MAX_SHIFT_PX:
            H_c = C @ H_coarse_geo
            if abs(np.linalg.det(H_c[:2, :2])) > 1e-9:
                H_fine0 = H_at_level(H_c, TARGET_GSD_COARSE_M, TARGET_GSD_FINE_M)
                coarse_model, REFINED = f'geometric prior + {_m} correction', True
                n_sg_in    = int(coarse_mask[pool_src == 'superglue'].sum())
                n_rift2_in = int(coarse_mask[pool_src == 'rift2'].sum())
                print(f"  ACCEPTED -> {coarse_model}")
        else:
            print(f"  REJECTED (need >={REFINE_MIN_INLIERS} inliers and <="
                  f"{REFINE_MAX_SHIFT_PX} px shift) — keeping the geometric prior.")
    else:
        print("  no usable correction model — keeping the geometric prior.")
else:
    print("  too few matches to attempt a correction — keeping the geometric prior.")

if not REFINED:
    print("\nProceeding on the GEOMETRIC PRIOR alone. That is a legitimate starting point:\n"
          "  it is exact at the corners and only degrades between them. The tiled stage\n"
          "  below refines it locally, which is where the accuracy comes from anyway.")

# The prior is affine and invertible by construction, but guard anyway — a singular H_fine0
# is what crashed the tiled stage before.
if not np.all(np.isfinite(H_fine0)) or abs(np.linalg.det(H_fine0[:2, :2])) < 1e-12:
    raise RuntimeError("H_fine0 is singular or non-finite. This should be impossible with "
                       "the geometric prior — check the two corner models.")

H_coarse_nac = H_coarse_geo
# Which matchers earn a place in the (expensive) fine stage.
# 'dense' is always included: the band check above proves dense correlation works on this
# pair, and on a low-sun scene it is often the ONLY thing that works at tile scale.
FINE_MATCHERS = ['dense'] + (['superglue'] if USE_SUPERGLUE_IN_TILES else [])
if USE_RIFT2 and n_rift2_in > 0 and n_rift2_in >= 0.25 * max(1, n_sg_in + n_rift2_in):
    FINE_MATCHERS.append('rift2')
print(f"\nH_fine0 ready ({coarse_model}).  FINE_MATCHERS = {FINE_MATCHERS}")


## Fine stage — tiled matching at native resolution

This is where the accuracy is. Matching two whole strips capped to a couple of megapixels
means registering at ~6.5 m/px — a 25× downsample of OHRC's native 0.26 m, with nearly all
the texture the matcher needs gone before it runs.

Instead: cut overlapping tiles at the fine GSD, use the coarse transform to pre-warp each
OHRC tile into its NAC counterpart's frame so the two are *nearly aligned* before matching
(the regime SuperGlue was trained for), match, then map the correspondences back into a
common frame and pool them.

Three things fall out for free. Memory stays bounded, because tiles are pulled lazily from
the memmap and the fine-resolution image is never materialised. Spatial coverage improves
by construction, because matches come from across the whole overlap rather than one lucky
patch. And each tile gets its own local contrast normalisation, which is far better than one
global stretch over a 25 km strip.

**v4 adds a texture screen.** At 81° incidence a lot of tiles are pure shadow with nothing
to match. Scoring them with a Laplacian variance first and skipping the dead ones costs
microseconds and saves a full model forward pass each.

In [ ]:
# 13) Tiled matching guided by the coarse transform
TILE           = SG_SIZE_FINE   # 640 — matches the processor size, so tiles are not resampled
TILE_OVERLAP   = 0.25
SEARCH_MARGIN  = 128            # Slack around the predicted NAC window. The dense check in
                                # the coarse cell MEASURES the prior's residual (~70 fine px
                                # on this pair), so 128 brackets it with room to spare.
                                # BIGGER IS NOT SAFER: the window goes to a fixed-size
                                # processor, so a wider window is a COARSER look at the same
                                # ground. At 320 the window was ~1350 px fed to a 640 input,
                                # halving the resolution, and every tile matched noise.
SG_TILE_INPUT  = 1024           # must be >= the window (708 + 2*128 = 964) or SuperGlue
                                # silently downsamples the tile
MIN_VALID_FRAC = 0.25           # skip a tile pair whose warped overlap is mostly empty
MIN_TEXTURE    = 12.0           # Laplacian variance floor — skip dead/shadowed tiles
# Per-tile consensus. Inside one 640 px tile the residual transform after the pre-warp is
# essentially a TRANSLATION, so genuine matches all agree on nearly the same displacement
# and noise does not. Filtering here rather than pooling everything into one global RANSAC
# matters enormously: ~60 tiles x ~57 matches with a handful real per tile is a 5-9% global
# inlier fraction, and RANSAC needs 70,000-700,000 iterations to find a 4-point model at
# that rate. maxIters is 20,000. The consensus never gets found. Per tile the same data is
# 100% separable.
TILE_CONSENSUS_PX   = 6.0       # agreement radius on the displacement, fine px
MIN_TILE_MATCHES    = 5         # a tile contributes nothing below this

stride = max(1, int(TILE * (1.0 - TILE_OVERLAP)))
if not np.all(np.isfinite(H_fine0)) or abs(np.linalg.det(H_fine0[:2, :2])) < 1e-12:
    raise RuntimeError(
        "H_fine0 is singular — the coarse stage produced a degenerate transform and this "
        "cell cannot invert it. Look at the coarse output: a similarity fit from a handful "
        "of inliers can collapse. With the geometric prior in place this should not happen.")
H_fine0_inv = np.linalg.inv(H_fine0)

fine_pts_o, fine_pts_n, fine_scores, fine_src = [], [], [], []
tiles_tried = tiles_used = tiles_skipped_texture = tiles_skipped_overlap = 0
tiles_no_consensus = 0
tile_shift, tile_kept = [], []
_t_start = time.time()

_rows = list(range(0, max(1, OHRC_FINE_SHAPE[0] - TILE // 2), stride))
_cols = list(range(0, max(1, OHRC_FINE_SHAPE[1] - TILE // 2), stride))
print(f'Tile grid over OHRC fine extent {OHRC_FINE_SHAPE}: {len(_rows)} x {len(_cols)} = '
      f'{len(_rows)*len(_cols)} positions (cap MAX_TILES={MAX_TILES})')

_stop = False
for r0 in _rows:
    if _stop: break
    for c0 in _cols:
        if tiles_tried >= MAX_TILES:
            _stop = True; break
        tiles_tried += 1

        o_tile_raw, o_origin = level_tile(ohrc_crop, OHRC_NATIVE_GSD_M, TARGET_GSD_FINE_M,
                                          r0, c0, TILE, TILE)
        if o_tile_raw is None:
            continue
        ox, oy = o_origin
        oh, ow = o_tile_raw.shape[:2]

        # Cheap screen BEFORE any model call: is there anything here to match?
        if texture_score(to_uint8(o_tile_raw)) < MIN_TEXTURE:
            tiles_skipped_texture += 1
            continue

        corners = np.array([[ox, oy], [ox + ow, oy], [ox, oy + oh], [ox + ow, oy + oh]])
        # LOCAL prior: the shift field measured band by band in the coarse cell, evaluated
        # at this tile's own position. A single global shift leaves the drift behind.
        _py = apply_h(H_fine0, [[ox + ow / 2.0, oy + oh / 2.0]])[0][1]
        _sx, _sy = prior_shift_fine(_py)
        H_local = mat_translate(_sx, _sy) @ H_fine0
        H_local_inv = np.linalg.inv(H_local)
        proj = apply_h(H_local, corners)
        n_c0 = int(np.floor(proj[:, 0].min())) - SEARCH_MARGIN
        n_r0 = int(np.floor(proj[:, 1].min())) - SEARCH_MARGIN
        n_w  = int(np.ceil(proj[:, 0].max() - proj[:, 0].min())) + 2 * SEARCH_MARGIN
        n_h  = int(np.ceil(proj[:, 1].max() - proj[:, 1].min())) + 2 * SEARCH_MARGIN
        if n_w < 32 or n_h < 32 or n_r0 > NAC_FINE_SHAPE[0] or n_c0 > NAC_FINE_SHAPE[1]:
            continue

        n_tile_raw, n_origin = level_tile(nac_crop, NAC_NATIVE_GSD_M, TARGET_GSD_FINE_M,
                                          max(0, n_r0), max(0, n_c0), n_h, n_w)
        if n_tile_raw is None:
            continue
        nx, ny = n_origin
        nh, nw = n_tile_raw.shape[:2]
        if max(nh, nw) > SG_TILE_INPUT and tiles_tried == 1:
            print(f'  *** WINDOW {nh}x{nw} EXCEEDS SG_TILE_INPUT={SG_TILE_INPUT}: SuperGlue is '
                  f'downsampling\n      every tile. Lower SEARCH_MARGIN or raise '
                  f'SG_TILE_INPUT before trusting anything below. ***')

        # ohrc tile local px -> ohrc fine -> nac fine -> nac tile local px
        H_tile = mat_translate(-nx, -ny) @ H_local @ mat_translate(ox, oy)
        valid = cv2.warpPerspective(np.full((oh, ow), 255, np.uint8), H_tile, (nw, nh),
                                    flags=cv2.INTER_NEAREST, borderValue=0)
        if valid.mean() / 255.0 < MIN_VALID_FRAC:
            tiles_skipped_overlap += 1
            continue

        warped_o_raw = cv2.warpPerspective(np.asarray(o_tile_raw), H_tile, (nw, nh),
                                           flags=cv2.INTER_LINEAR, borderValue=0)
        # FIX C5: stretch the warped tile using ONLY the pixels that came from real data.
        # Percentiles over the black border flatten the real range before the matcher sees it.
        vmask = valid > 0
        o_pc, n_pc, o_sg, n_sg = prep_pair(warped_o_raw, n_tile_raw, mask=vmask)

        if texture_score(n_pc) < MIN_TEXTURE:
            tiles_skipped_texture += 1
            continue

        back_to_ohrc = H_local_inv @ mat_translate(nx, ny)
        to_nac_fine  = mat_translate(nx, ny)

        got = 0
        for name in FINE_MATCHERS:
            if name == 'superglue':
                ka, kb, sc = match_superglue(o_sg, n_sg, size=SG_TILE_INPUT)
            elif name == 'dense':
                ka, kb, sc = match_dense(o_pc, n_pc, vmask)
            else:
                ka, kb, sc = match_rift2(o_pc, n_pc)
            if len(ka) == 0:
                continue
            xi = np.clip(ka[:, 0].astype(int), 0, nw - 1)
            yi = np.clip(ka[:, 1].astype(int), 0, nh - 1)
            keep = valid[yi, xi] > 0        # drop matches landing on the warp's black border
            ka, kb, sc = ka[keep], kb[keep], sc[keep]
            _min_here = 2 if name == 'dense' else MIN_TILE_MATCHES
            if len(ka) < _min_here:
                continue

            # --- per-tile consensus on the residual displacement -------------------------
            d = kb - ka
            med = np.median(d, axis=0)
            dev = np.linalg.norm(d - med, axis=1)
            mad = float(np.median(dev))
            # The tile's own matches must AGREE, full stop. An adaptive threshold that grows
            # with the scatter is self-defeating: when everything disagrees it widens until
            # it accepts everything. That is how 88% of pure noise once got through here.
            if mad > TILE_CONSENSUS_PX:
                tiles_no_consensus += 1
                continue
            if name == 'dense' and len(ka) < 3:
                mad = 0.0          # too few points for a meaningful spread; the NCC null
                                   # margin inside match_dense is the gate for these
            agree = dev <= TILE_CONSENSUS_PX
            if int(agree.sum()) < _min_here:
                tiles_no_consensus += 1
                continue
            ka, kb, sc = ka[agree], kb[agree], sc[agree]
            tile_shift.append(np.median(kb - ka, axis=0))
            tile_kept.append((int(agree.sum()), int(len(agree))))

            fine_pts_o.append(apply_h(back_to_ohrc, ka))
            fine_pts_n.append(apply_h(to_nac_fine, kb))
            fine_scores.append(sc)
            fine_src.extend([name] * len(ka))
            got += len(ka)

        if got:
            tiles_used += 1
        if tiles_tried % 20 == 0:
            print(f'  {tiles_tried:4d} tried | {tiles_used:4d} matched | '
                  f'{tiles_skipped_texture:3d} no-texture | {tiles_skipped_overlap:3d} no-overlap | '
                  f'{sum(len(p) for p in fine_pts_o):6d} pts | {time.time()-_t_start:.0f}s')

if not fine_pts_o:
    print('\n*** No tile produced matches. Falling back to the coarse correspondences. ***')
    FINE_STAGE_OK = False
    _c2f = mat_scale(TARGET_GSD_COARSE_M / TARGET_GSD_FINE_M)
    fine_o = apply_h(_c2f, pool0)
    fine_n = apply_h(_c2f @ np.linalg.inv(M_var_best), pool1)
    fine_s, fine_m = pools, pool_src
else:
    FINE_STAGE_OK = True
    fine_o = np.vstack(fine_pts_o)
    fine_n = np.vstack(fine_pts_n)
    fine_s = np.concatenate(fine_scores)
    fine_m = np.array(fine_src)

if tile_shift:
    _ts = np.asarray(tile_shift)
    _kept = np.asarray(tile_kept)
    print(f'\nPer-tile consensus: kept {_kept[:,0].sum()} of {_kept[:,1].sum()} raw matches '
          f'({100.0*_kept[:,0].sum()/max(_kept[:,1].sum(),1):.0f}%) across {len(_ts)} tiles')
    print(f'  residual displacement of the LOCAL prior, per tile'
          f'{" (shift field applied)" if PRIOR_SHIFT_COEF is not None else ""}:')
    print(f'    median ({np.median(_ts[:,0]):+.1f}, {np.median(_ts[:,1]):+.1f}) fine px'
          f'   = ({np.median(_ts[:,0])*TARGET_GSD_FINE_M:+.0f}, '
          f'{np.median(_ts[:,1])*TARGET_GSD_FINE_M:+.0f}) m')
    print(f'    spread  ({_ts[:,0].std():.1f}, {_ts[:,1].std():.1f}) px'
          f'   max |shift| {np.abs(_ts).max():.0f} px')
    if np.abs(_ts).max() > SEARCH_MARGIN * 0.8:
        print(f'    *** Some tiles needed nearly the whole {SEARCH_MARGIN} px search margin.')
        print(f'        The prior is drifting more than the window allows — raise')
        print(f'        SEARCH_MARGIN and re-run, or the far end of the strip is matching noise. ***')
    elif _ts.std(axis=0).max() > 40:
        print(f'    (a large spread means the affine corner model does not capture the real')
        print(f'     pushbroom geometry — expected with non-zero roll/pitch; the tiled fit')
        print(f'     absorbs it, which is exactly why this stage exists)')

print(f'\nFine stage: {tiles_used}/{tiles_tried} tiles matched '
      f'({tiles_skipped_texture} texture, {tiles_skipped_overlap} overlap, '
      f'{tiles_no_consensus} no-consensus), '
      f'{len(fine_o)} correspondences in {time.time()-_t_start:.0f}s')
for name in sorted(set(fine_m.tolist())):
    print(f'  {name}: {int((fine_m == name).sum())}')

In [ ]:
# 14) Final model on the pooled fine correspondences
# is_degenerate's 0.02 default assumes a roughly square overlap. This pair's overlap is a
# long narrow strip (~25 km x ~4.8 km, eigenvalue ratio ~(4.8/25)^2 ~ 0.037), so a CORRECT
# set of inliers spread evenly along it sits near that threshold and can be rejected as
# "collinear" when it is nothing of the kind. Diagnostics print BEFORE any rejection, so a
# genuine failure (matches bunched at one end) stays distinguishable from this legitimate
# narrow-overlap case rather than being masked by the relaxed threshold.
FINE_MIN_MINOR_AXIS_PX = 5.0   # perpendicular spread, in fine px

def _diagnose(pts, frame_shape, label):
    pts = np.asarray(pts, np.float64).reshape(-1, 2)
    if len(pts) < 3:
        print(f"  {label}: fewer than 3 points"); return
    lo, hi = pts.min(0), pts.max(0)
    print(f"  {label}: {len(pts)} pts | x {lo[0]:.0f}-{hi[0]:.0f}, y {lo[1]:.0f}-{hi[1]:.0f} px"
          f" | minor axis {minor_axis_px(pts):.1f} px"
          f" | hull {hull_coverage(pts, frame_shape):.2%}")

print("Geometry of the correspondences:")
_diagnose(fine_n, NAC_FINE_SHAPE, "all pooled")

# --- The RANSAC threshold must match the data, not a hopeful constant --------------------
# RANSAC_THRESH_PX = 3 assumes the two images are related by a 2D transform to within 3 px.
# They are not. OHRC views at roll -6.2 deg / pitch +17.4 deg, NAC near-nadir at emission
# 1.7 deg: a ~17 deg look-angle difference. Relief-induced parallax is then ~0.3 m of
# displacement per metre of local relief, so even 100 m of topography moves a feature ~30 m
# = ~33 fine px between the two views. No 2D transform can remove that; only a DTM can.
#
# Left at 3 px, RANSAC cannot find a real consensus and locks onto whatever tiny accidental
# subset agrees -- 16 collinear points, which the degeneracy guard then rightly refuses.
# So: estimate the residual scale FROM THE DATA and set the threshold from it.
AUTO_THRESHOLD   = True
THRESH_SIGMA_MUL = 2.5      # keep points within this many robust sigmas of the model

def estimate_residual_scale(src, dst, frame_shape, t0=100.0, iters=3):
    """Robust scale of the fit residual, in px.

    Two traps this avoids:
      * fitting with an effectively infinite threshold makes RANSAC degenerate to least
        squares over inliers AND outliers, so the model is dragged off and the "scale" comes
        out hundreds of px (measured: 164-777 px on 15-40% outliers);
      * taking the MAD over the fit's own INLIERS underestimates, because those points were
        selected for being close. Measured: stuck at the 3 px floor whatever the truth.
    So: fit robustly at a moderate threshold, then take the MAD over ALL residuals. MAD has
    a 50% breakdown point, so it survives a large outlier fraction. Verified to track the
    ideal threshold within ~20% up to 25% outliers.
    """
    t = t0
    for _ in range(iters):
        H, m, _ = robust_fit(src, dst, frame_shape, thresh=t, verbose=False,
                             min_minor_axis_px=0.0)
        if H is None:
            return None
        r = np.linalg.norm(apply_h(H, src) - dst, axis=1)
        s_ = 1.4826 * float(np.median(np.abs(r - np.median(r))))
        t = float(max(3.0, THRESH_SIGMA_MUL * max(s_, 1e-3)))
    return t

if AUTO_THRESHOLD:
    _t = estimate_residual_scale(fine_o, fine_n, NAC_FINE_SHAPE)
    if _t is not None:
        FIT_THRESH_PX = _t
        print(f"\nResidual scale from the data: {_t/THRESH_SIGMA_MUL:.1f} fine px "
              f"({_t/THRESH_SIGMA_MUL*TARGET_GSD_FINE_M:.1f} m)")
        print(f"  -> RANSAC threshold {FIT_THRESH_PX:.1f} fine px "
              f"(fixed default was {RANSAC_THRESH_PX:.1f})")
    else:
        FIT_THRESH_PX = RANSAC_THRESH_PX
        print(f"\nCould not estimate a residual scale; using {FIT_THRESH_PX:.1f} px")
else:
    FIT_THRESH_PX = RANSAC_THRESH_PX

# --- Is that residual STRUCTURE or NOISE? -----------------------------------------------
# Decisive and free: within one tile the matches agree to better than TILE_CONSENSUS_PX,
# because that is the gate they passed. If BETWEEN tiles they disagree by far more, the
# disagreement is a function of position or terrain, not random matching error.
try:
    _between = float(np.mean(np.asarray(tile_shift).std(axis=0)))
    print(f"\nResidual structure: within-tile agreement <= {TILE_CONSENSUS_PX:.0f} px "
          f"(the consensus gate), between-tile spread {_between:.1f} px")
    if _between > 3 * TILE_CONSENSUS_PX:
        print(f"  -> x{_between/TILE_CONSENSUS_PX:.0f} larger between tiles than within one.")
        print( "     This is NOT matcher noise. It is real deformation the global model")
        print( "     cannot represent — relief parallax between two look angles, and/or")
        print( "     pushbroom geometry an affine corner model cannot capture.")
        print( "     Registering to better than this needs a DTM, not a better matcher.")
except (NameError, ValueError):
    pass

H_fine, inlier_mask, MODEL_NAME = robust_fit(fine_o, fine_n, NAC_FINE_SHAPE,
                                             thresh=FIT_THRESH_PX,
                                             min_minor_axis_px=FINE_MIN_MINOR_AXIS_PX)
if H_fine is None:
    _H2, _m2, _n2 = robust_fit(fine_o, fine_n, NAC_FINE_SHAPE, thresh=FIT_THRESH_PX,
                               verbose=False, min_minor_axis_px=0.0)
    if _m2 is not None:
        _diagnose(fine_n[_m2], NAC_FINE_SHAPE, "the INLIERS that were rejected")
    raise RuntimeError(
        "The pooled fine correspondences did not support any model even at a data-driven\n"
        f"  threshold of {FIT_THRESH_PX:.1f} px. If the rejected inliers are a handful of\n"
        "  collinear points, RANSAC still found no real consensus — check the per-tile\n"
        "  spread in the previous cell.")

_diagnose(fine_n[inlier_mask], NAC_FINE_SHAPE, "inliers")

n_inliers    = int(inlier_mask.sum())
inlier_ratio = n_inliers / len(fine_o)
coverage     = hull_coverage(fine_n[inlier_mask], NAC_FINE_SHAPE)

print(f'\nModel: {MODEL_NAME}')
print(f'Inliers: {n_inliers} / {len(fine_o)}  ({inlier_ratio:.1%})')
print(f'Spatial coverage: {coverage:.2%} of the NAC frame')
print('Per-matcher contribution (this is the RIFT2 ablation answer):')
for name in sorted(set(fine_m.tolist())):
    sel = (fine_m == name)
    cov_n = hull_coverage(fine_n[sel & inlier_mask], NAC_FINE_SHAPE)
    print(f'   {name:10s} {int((sel & inlier_mask).sum()):5d} inliers of {int(sel.sum()):5d}'
          f'   (their own hull coverage {cov_n:.2%})')

TRUSTED = (n_inliers >= MIN_INLIERS_TO_TRUST) and (coverage >= MIN_COVERAGE_TO_TRUST)
if not TRUSTED:
    print(f'\n*** LOW CONFIDENCE: {n_inliers} inliers over {coverage:.2%} of the frame '
          f'(need >={MIN_INLIERS_TO_TRUST} and >={MIN_COVERAGE_TO_TRUST:.0%}). '
          'Treat the warp below as exploratory. ***')

In [ ]:
# 15) ECC refinement — gated
# Feature matching alone will not reach sub-pixel accuracy across an illumination gap. Once
# a coarse-to-fine model exists an intensity-based refinement is cheap, and gradient
# magnitude survives an illumination change far better than brightness does.
#
# The caveat this cell takes seriously: the refinement runs on whole images at a COARSER
# level than the tiled fit, so it can only resolve to about that level's pixel. If the tiled
# fit is already sub-pixel at the fine level, an unconditional ECC step makes the answer
# WORSE — verified on synthetic data, where a 0.05 fine-px fit was pushed to 1.04 by an
# ungated coarse ECC. So each candidate is scored at the FINE level on real tiles and kept
# only if it actually improves. Generally: a refinement coarser than your current solution
# is a downgrade dressed as an improvement.
ECC_ITERATIONS, ECC_EPS = 200, 1e-6
ECC_PIXEL_BUDGET, ECC_EVAL_TILES, ECC_EVAL_SIZE = 6_000_000, 6, 640

def ecc_refine(H_in, gsd):
    o_g = grad_mag_u8(to_uint8(level_view(ohrc_crop, OHRC_NATIVE_GSD_M, gsd)))
    n_g = grad_mag_u8(to_uint8(level_view(nac_crop,  NAC_NATIVE_GSD_M,  gsd)))
    W = H_at_level(H_in, TARGET_GSD_FINE_M, gsd).astype(np.float32)
    motion = cv2.MOTION_HOMOGRAPHY if MODEL_NAME == 'homography' else cv2.MOTION_AFFINE
    warp = W.astype(np.float32) if motion == cv2.MOTION_HOMOGRAPHY else W[:2].astype(np.float32)
    crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, ECC_ITERATIONS, ECC_EPS)
    # findTransformECC's warpMatrix maps TEMPLATE -> INPUT. With OHRC as template and NAC as
    # input that is exactly our OHRC -> NAC transform, so no inversion is needed.
    cc, warp = cv2.findTransformECC(o_g, n_g, warp, motion, crit, None, 5)
    W_ref = warp if motion == cv2.MOTION_HOMOGRAPHY else to_h(warp)
    return H_at_level(W_ref, gsd, TARGET_GSD_FINE_M), float(cc)

def fine_alignment_score(H, n_tiles=ECC_EVAL_TILES, size=ECC_EVAL_SIZE):
    """Mean ECC correlation over FINE-level tiles, NAC rendered into each OHRC tile's frame."""
    rows = np.linspace(0, max(1, OHRC_FINE_SHAPE[0] - size), max(2, n_tiles)).astype(int)
    cols = np.linspace(0, max(1, OHRC_FINE_SHAPE[1] - size), 2).astype(int)
    vals = []
    for r0 in rows:
        for c0 in cols:
            o_tile, o_org = level_tile(ohrc_crop, OHRC_NATIVE_GSD_M, TARGET_GSD_FINE_M,
                                       int(r0), int(c0), size, size)
            if o_tile is None: continue
            ox, oy = o_org; oh, ow = o_tile.shape[:2]
            corners = np.array([[ox, oy], [ox + ow, oy], [ox, oy + oh], [ox + ow, oy + oh]])
            proj = apply_h(H, corners)
            n_r0 = int(np.floor(proj[:, 1].min())) - 8
            n_c0 = int(np.floor(proj[:, 0].min())) - 8
            n_h  = int(np.ceil(proj[:, 1].max() - proj[:, 1].min())) + 16
            n_w  = int(np.ceil(proj[:, 0].max() - proj[:, 0].min())) + 16
            if n_h < 32 or n_w < 32: continue
            n_tile, n_org = level_tile(nac_crop, NAC_NATIVE_GSD_M, TARGET_GSD_FINE_M,
                                       max(0, n_r0), max(0, n_c0), n_h, n_w)
            if n_tile is None: continue
            nx, ny = n_org
            H_tile = mat_translate(-nx, -ny) @ H @ mat_translate(ox, oy)
            n_in_o = cv2.warpPerspective(to_uint8(n_tile), H_tile.astype(np.float32), (ow, oh),
                                         flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP,
                                         borderValue=0)
            m = (n_in_o > 0).astype(np.uint8) * 255
            if m.mean() / 255.0 < 0.5: continue
            try:
                vals.append(float(cv2.computeECC(grad_mag_u8(to_uint8(o_tile)),
                                                 grad_mag_u8(n_in_o), m)))
            except cv2.error:
                continue
            if len(vals) >= n_tiles: break
        if len(vals) >= n_tiles: break
    return float(np.mean(vals)) if vals else None

H_before_ecc = H_fine.copy()
ECC_APPLIED, ECC_CC = False, None
score_before = fine_alignment_score(H_fine) if RUN_ECC else None
print('Fine-level alignment score before refinement:',
      f'{score_before:.4f}' if score_before is not None else 'n/a')

if RUN_ECC and score_before is not None:
    ladder, g = [TARGET_GSD_COARSE_M], TARGET_GSD_COARSE_M / 2.0
    while g > TARGET_GSD_FINE_M * 0.99 and \
          max(_px_at(ohrc_crop.shape, OHRC_NATIVE_GSD_M, g),
              _px_at(nac_crop.shape, NAC_NATIVE_GSD_M, g)) <= ECC_PIXEL_BUDGET:
        ladder.append(g); g /= 2.0

    best_H, best_score, best_g = H_fine, score_before, None
    for g in ladder:
        try:
            H_try, cc = ecc_refine(H_fine, g)
        except cv2.error as e:
            print(f'  {g:6.2f} m/px: did not converge ({str(e).splitlines()[-1][:60]})'); continue
        if not np.all(np.isfinite(H_try)):
            continue
        s = fine_alignment_score(H_try)
        print(f'  {g:6.2f} m/px: ECC cc={cc:.4f}, fine score '
              f'{"n/a" if s is None else round(s,4)} -> '
              f'{"kept" if (s is not None and s > best_score) else "rejected"}')
        if s is not None and s > best_score:
            best_H, best_score, best_g, ECC_CC = H_try, s, g, cc

    if best_g is not None:
        shift = float(np.linalg.norm(apply_h(best_H, [[0, 0]]) - apply_h(H_before_ecc, [[0, 0]])))
        H_fine, ECC_APPLIED = best_H, True
        print(f'\nECC accepted from the {best_g:.2f} m/px pass: {score_before:.4f} -> '
              f'{best_score:.4f}, origin moved {shift:.2f} fine px '
              f'({shift*TARGET_GSD_FINE_M:.2f} m).')
    else:
        print('\nNo ECC pass improved the fine score — keeping the RANSAC transform. '
              'That is a normal, good outcome when the tiled fit is already sub-pixel.')
else:
    print('\nECC skipped.')

ECC_FINE_SCORE = fine_alignment_score(H_fine)

In [ ]:
# 16) Visualise matches, warp and overlay
def plot_keypoint_matches(img1, img2, kpts0, kpts1, scores, max_matches=200,
                          save_path=None, title_prefix=""):
    if len(kpts0) == 0:
        print(title_prefix, '(no matches to draw)'); return
    order = np.argsort(-scores)[:max_matches]
    k0, k1, s = kpts0[order], kpts1[order], scores[order]
    h1, w1 = img1.shape[:2]; h2, w2 = img2.shape[:2]
    canvas = np.full((max(h1, h2), w1 + w2), 255, np.uint8)
    canvas[:h1, :w1] = img1; canvas[:h2, w1:w1 + w2] = img2
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.imshow(canvas, cmap="gray")
    cmap = plt.get_cmap("winter")
    for (x0, y0), (x1, y1), sc in zip(k0, k1, s):
        col = cmap(float(np.clip(sc, 0, 1)))
        ax.plot([x0, x1 + w1], [y0, y1], "-", color=col, linewidth=0.8, alpha=0.8)
        ax.plot(x0, y0, "o", color=col, markersize=3)
        ax.plot(x1 + w1, y1, "o", color=col, markersize=3)
    ax.set_title(f"{title_prefix}{len(k0)} shown (scores {s.min():.2f}-{s.max():.2f})")
    ax.axis("off"); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

_f2c = TARGET_GSD_FINE_M / TARGET_GSD_COARSE_M
plot_keypoint_matches(ohrc_c_pc, nac_c_pc, fine_o * _f2c, fine_n * _f2c, fine_s,
                      save_path=os.path.join(OUT_DIR, f'{PAIR_ID}_v4_02_all_matches.png'),
                      title_prefix='Pooled fine matches (pre-RANSAC), at the coarse level: ')
plot_keypoint_matches(ohrc_c_pc, nac_c_pc, (fine_o * _f2c)[inlier_mask],
                      (fine_n * _f2c)[inlier_mask], fine_s[inlier_mask],
                      save_path=os.path.join(OUT_DIR, f'{PAIR_ID}_v4_03_inliers.png'),
                      title_prefix=f'RANSAC inliers ({MODEL_NAME}): ')

H_vis = H_at_level(H_fine, TARGET_GSD_FINE_M, TARGET_GSD_COARSE_M)
h_ref, w_ref = nac_c_pc.shape
warped = cv2.warpPerspective(ohrc_c_pc, H_vis, (w_ref, h_ref))

overlay = np.zeros((h_ref, w_ref, 3), np.uint8)
overlay[..., 1] = nac_c_pc      # green = NAC
overlay[..., 0] = warped        # red   = warped OHRC
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(warped, cmap="gray");  ax[0].set_title("OHRC warped into NAC frame")
ax[1].imshow(nac_c_pc, cmap="gray"); ax[1].set_title("NAC reference")
ax[2].imshow(overlay);               ax[2].set_title("Overlay (green=NAC, red=OHRC)")
for a in ax: a.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"{PAIR_ID}_v4_04_overlay.png"), dpi=150)
plt.show()

# The checkerboard is the honest test — a red/green overlay can hide a misalignment that
# a discontinuity across a tile boundary makes obvious.
_blk = max(16, min(h_ref, w_ref) // 24)
_yy, _xx = np.mgrid[0:h_ref, 0:w_ref]
_mosaic = np.where((((_yy // _blk) + (_xx // _blk)) % 2).astype(bool), warped, nac_c_pc)
plt.figure(figsize=(9, 9))
plt.imshow(_mosaic, cmap='gray')
plt.title('Checkerboard: features should run continuously across every boundary')
plt.axis('off'); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'{PAIR_ID}_v4_05_checkerboard.png'), dpi=150)
plt.show()

In [ ]:
# 17) Compose back to native pixels and to lat/lon
H_native = np.linalg.inv(M_nac_native_to_fine) @ H_fine @ M_ohrc_native_to_fine
print('H (native OHRC pixel -> native NAC pixel):\n', H_native)

def ohrc_pixel_to_latlon(row, col):
    x, y = apply_h(H_native, [[col, row]])[0]
    return nac_fwd(y, x)

_h_o, _w_o = ohrc_shape
print()
for _name, (_r, _c) in [('centre', (_h_o // 2, _w_o // 2)), ('upper-left', (0, 0)),
                        ('lower-right', (_h_o - 1, _w_o - 1))]:
    _lat, _lon = ohrc_pixel_to_latlon(_r, _c)
    _dlat, _dlon = ohrc_fwd(_r, _c)
    _dx = (float(unwrap_lon(_lon, _dlon)) - _dlon) * km_per_deg_lon(_lat)
    _dy = (_lat - _dlat) * KM_PER_DEG_LAT
    print(f'  OHRC {_name:12s} ({_r:6d},{_c:6d}) -> {_lat:9.5f} N, {_lon:10.5f} E'
          f'  | label {_dlat:9.5f} N, {_dlon:10.5f} E  | offset {np.hypot(_dx,_dy)*1000:7.0f} m')

print('\nThose offsets are the correction the registration applies to the OHRC label '
      'geometry.\nA few hundred metres is plausible for SPICE-level pointing. Kilometres '
      'means the model\nis wrong however good the inlier count looks.')

np.save(os.path.join(OUT_DIR, f'{PAIR_ID}_v4_H_native.npy'), H_native)
np.save(os.path.join(OUT_DIR, f'{PAIR_ID}_v4_H_fine.npy'), H_fine)

In [ ]:
# 18) Evaluation — three independent checks
# Computing RMSE on the same inliers that fitted H is optimistic by construction, and with
# few points it is exactly zero. That is how v1 reported rmse_px = 0.0 beside a broken warp.
from sklearn.metrics import normalized_mutual_info_score

src_in, dst_in = fine_o[inlier_mask], fine_n[inlier_mask]

# 1. in-sample RMSE — reported, but labelled
rmse_in = float(np.sqrt(np.mean(np.sum((apply_h(H_fine, src_in) - dst_in) ** 2, axis=1))))

# 2. held-out RMSE — fit on 70%, measure on the 30% the fit never saw
rng = np.random.default_rng(0)
held = []
for _ in range(5):
    idx = rng.permutation(len(src_in)); cut = int(0.7 * len(idx))
    tr, te = idx[:cut], idx[cut:]
    if len(tr) < 8 or len(te) < 4: continue
    H_tr, _, _ = robust_fit(src_in[tr], dst_in[tr], NAC_FINE_SHAPE, verbose=False,
                            thresh=FIT_THRESH_PX,
                            min_minor_axis_px=FINE_MIN_MINOR_AXIS_PX)
    if H_tr is None: continue
    held.append(float(np.sqrt(np.mean(
        np.linalg.norm(apply_h(H_tr, src_in[te]) - dst_in[te], axis=1) ** 2))))
rmse_heldout = float(np.mean(held)) if held else None

# 3. feature-independent: NMI over the overlap, against a deliberately misaligned baseline
_valid = (warped > 0) & (nac_c_pc > 0)
if _valid.sum() > 1000:
    _sub = np.random.default_rng(0).choice(int(_valid.sum()),
                                           size=min(200_000, int(_valid.sum())), replace=False)
    nmi = float(normalized_mutual_info_score((warped[_valid] // 8)[_sub],
                                             (nac_c_pc[_valid] // 8)[_sub]))
    _shift = np.roll(warped, (37, 29), axis=(0, 1))
    _v2 = (_shift > 0) & (nac_c_pc > 0)
    _s2 = np.random.default_rng(0).choice(int(_v2.sum()),
                                          size=min(200_000, int(_v2.sum())), replace=False)
    nmi_baseline = float(normalized_mutual_info_score((_shift[_v2] // 8)[_s2],
                                                      (nac_c_pc[_v2] // 8)[_s2]))
else:
    nmi = nmi_baseline = None

metrics = {
    "pair_id": PAIR_ID, "version": "v4",
    "ohrc_product_id": ohrc_product_id, "nac_product_id": nac_product_id,
    "nac_corners_source": NAC_CORNERS_SOURCE,
    "use_rift2": bool(USE_RIFT2),
    "preflight_gate": {"verdict": gate["verdict"], "margin_over_null": round(gate["margin"], 3),
                       "null_floor": round(gate["floor"], 4),
                       "best_ncc": round(gate["best"][0], 4) if gate["best"] else None,
                       "best_flip": gate["best"][1] if gate["best"] else None,
                       "best_scale": gate["best"][3] if gate["best"] else None},
    "model": MODEL_NAME,
    "matchers_pooled": sorted(set(fine_m.tolist())), "fine_matchers_run": FINE_MATCHERS,
    "orientation": {"flip_code": BEST_FLIP_CODE, "rotation_deg": float(BEST_ROT_DEG),
                    "rotation_self_test_passed": bool(ROTATION_ESTIMATOR_TRUSTED)},
    "gsd": {"ohrc_native_m": OHRC_NATIVE_GSD_M, "nac_native_m": NAC_NATIVE_GSD_M,
            "fine_m": TARGET_GSD_FINE_M, "coarse_m": TARGET_GSD_COARSE_M},
    "crop": {"pad_m": PAD_M, "ohrc_window": list(ohrc_win), "nac_window": list(nac_win),
             "ohrc_pct_kept": round(ohrc_kept, 2), "nac_pct_kept": round(nac_kept, 2)},
    "tiles": {"tried": tiles_tried, "matched": tiles_used, "tile_px": TILE,
              "skipped_texture": tiles_skipped_texture,
              "skipped_overlap": tiles_skipped_overlap},
    "n_correspondences": int(len(fine_o)), "n_inliers": n_inliers,
    "inlier_ratio": float(inlier_ratio), "spatial_coverage_pct": float(coverage * 100.0),
    "rmse_px_in_sample": rmse_in, "rmse_m_in_sample": rmse_in * TARGET_GSD_FINE_M,
    "rmse_px_heldout": rmse_heldout,
    "rmse_m_heldout": (rmse_heldout * TARGET_GSD_FINE_M) if rmse_heldout else None,
    "heldout_splits": len(held),
    "nmi_aligned": nmi, "nmi_misaligned_baseline": nmi_baseline,
    "ecc_applied": bool(ECC_APPLIED), "ecc_correlation": ECC_CC,
    "fine_alignment_score": ECC_FINE_SCORE, "trusted": bool(TRUSTED),
}
print(json.dumps(metrics, indent=2, default=str))

print("\n--- how to read this ---")
print(f"in-sample RMSE  {rmse_in:.3f} px = {rmse_in*TARGET_GSD_FINE_M:.3f} m  (optimistic by construction)")
if rmse_heldout:
    print(f"held-out RMSE   {rmse_heldout:.3f} px = {rmse_heldout*TARGET_GSD_FINE_M:.3f} m  <- the honest number")
else:
    print("held-out RMSE   not computable — too few inliers to split. That is itself the result.")
if nmi is not None:
    print(f"NMI aligned {nmi:.4f} vs misaligned baseline {nmi_baseline:.4f} "
          f"({'clearly better' if nmi > nmi_baseline*1.15 else 'NOT convincingly better — the warp may be wrong'})")
print(f"coverage {coverage:.2%} of the frame from {n_inliers} inliers across {tiles_used} tiles")
if USE_RIFT2:
    print("\nRIFT2 ablation: compare the per-matcher inlier counts above against a re-run "
          "with USE_RIFT2 = False, on held-out RMSE and NMI — not on raw match counts.")

with open(os.path.join(OUT_DIR, f"{PAIR_ID}_v4_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"\nSaved -> {os.path.join(OUT_DIR, f'{PAIR_ID}_v4_metrics.json')}")

## Final outputs — sub-pixel correspondences and the warped raster

Everything up to here produces a *transform*. These two cells produce the deliverables:

1. **Sub-pixel refinement.** A raw SuperGlue match is good to about a pixel, because
   SuperPoint detects on a coarse grid. Sub-pixel accuracy is recovered afterwards by phase
   correlation on gradient magnitude, per correspondence, after rendering the OHRC
   neighbourhood into the NAC window so only a small translation remains. Gated twice: per
   point on correlation response, and globally on whether held-out RMSE actually improved.

2. **Export.** Every matched point in fine px, native px and lat/lon with its residual, as
   CSV and NPZ; and the OHRC image genuinely resampled into the NAC frame at full fine
   resolution, written in row bands, with a georeferencing sidecar.


In [ ]:
# 19) SUB-PIXEL REFINEMENT of the correspondences
#
# SuperPoint emits keypoints on a coarse grid with a soft-argmax, so a raw SuperGlue match is
# good to roughly a pixel — not better. Sub-pixel accuracy has to be recovered afterwards, by
# looking at the actual intensities around each match.
#
# Method, per correspondence:
#   1. take a window around the NAC point,
#   2. render the OHRC neighbourhood into that same window using the current model, so the
#      only thing left between them is a small translation,
#   3. measure that translation with phase correlation, which is sub-pixel by construction,
#   4. run it on GRADIENT MAGNITUDE, not intensity, so a 20 deg sun-elevation difference
#      does not dominate the correlation.
#
# Every step is gated: a correspondence is only updated when its correlation response clears
# a threshold and the correction is small, and the whole refinement is only kept if held-out
# RMSE actually improves. A refinement you cannot show improved something is not a refinement.
SUBPIX_WINDOW      = 96      # fine-level px
SUBPIX_MIN_RESPONSE = 0.06   # phase-correlation peak response
SUBPIX_MAX_SHIFT   = 3.0     # a "correction" bigger than this is a mis-match, not a refinement
SUBPIX_MAX_POINTS  = 4000    # time guard

def _fine_patch(raster, native_gsd, cx, cy, size):
    """Patch of `size` fine px centred on (cx, cy) in fine-level coords. (patch, x0, y0)."""
    half = size // 2
    r0, c0 = int(round(cy - half)), int(round(cx - half))
    tile, org = level_tile(raster, native_gsd, TARGET_GSD_FINE_M, r0, c0, size, size)
    if tile is None or min(tile.shape[:2]) < size // 2:
        return None, None, None
    return tile, org[0], org[1]

_win = cv2.createHanningWindow((SUBPIX_WINDOW, SUBPIX_WINDOW), cv2.CV_32F)

def refine_correspondences(src_pts, dst_pts, H):
    """Returns (dst_refined, ok_mask, shifts). src in ohrc-fine, dst in nac-fine."""
    dst_out = np.array(dst_pts, np.float64, copy=True)
    ok = np.zeros(len(src_pts), bool)
    shifts = np.zeros(len(src_pts))
    n = min(len(src_pts), SUBPIX_MAX_POINTS)
    order = np.argsort(-np.asarray(fine_s)[inlier_mask])[:n] if len(src_pts) > n else range(n)
    t0 = time.time()
    for count, i in enumerate(order):
        px, py = float(src_pts[i][0]), float(src_pts[i][1])
        qx, qy = float(dst_pts[i][0]), float(dst_pts[i][1])
        n_patch, nx0, ny0 = _fine_patch(nac_crop, NAC_NATIVE_GSD_M, qx, qy, SUBPIX_WINDOW)
        if n_patch is None:
            continue
        h, w = n_patch.shape[:2]
        # render OHRC into this NAC window using the current model
        H_win = mat_translate(-nx0, -ny0) @ H
        o_half = SUBPIX_WINDOW
        o_patch, ox0, oy0 = _fine_patch(ohrc_crop, OHRC_NATIVE_GSD_M, px, py, o_half * 2)
        if o_patch is None:
            continue
        H_local = H_win @ mat_translate(ox0, oy0)
        o_in_n = cv2.warpPerspective(o_patch, H_local.astype(np.float64), (w, h),
                                     flags=cv2.INTER_CUBIC, borderValue=0)
        valid = cv2.warpPerspective(np.full(o_patch.shape, 255, np.uint8),
                                    H_local.astype(np.float64), (w, h),
                                    flags=cv2.INTER_NEAREST, borderValue=0)
        if (valid > 0).mean() < 0.85:
            continue
        a = grad_mag_u8(to_uint8(o_in_n, mask=valid > 0)).astype(np.float32)
        b = grad_mag_u8(to_uint8(n_patch)).astype(np.float32)
        if a.shape != b.shape or min(a.shape) < 16:
            continue
        wfn = _win if a.shape == _win.shape else cv2.createHanningWindow(
            (a.shape[1], a.shape[0]), cv2.CV_32F)
        try:
            (dx, dy), resp = cv2.phaseCorrelate(a * wfn, b * wfn)
        except cv2.error:
            continue
        mag = float(np.hypot(dx, dy))
        if resp >= SUBPIX_MIN_RESPONSE and mag <= SUBPIX_MAX_SHIFT:
            dst_out[i] = [qx + dx, qy + dy]
            ok[i] = True
            shifts[i] = mag
        if count and count % 500 == 0:
            print(f"    {count}/{n} points, {int(ok.sum())} refined, {time.time()-t0:.0f}s")
    return dst_out, ok, shifts

print(f"Refining {int(inlier_mask.sum())} inlier correspondences "
      f"(window {SUBPIX_WINDOW} px at {TARGET_GSD_FINE_M:.3f} m)...")
_src_in, _dst_in = fine_o[inlier_mask], fine_n[inlier_mask]
_dst_ref, _ok, _sh = refine_correspondences(_src_in, _dst_in, H_fine)
print(f"  {int(_ok.sum())} of {len(_src_in)} correspondences refined "
      f"({100.0*_ok.mean():.0f}%), median correction {np.median(_sh[_ok]) if _ok.any() else 0:.3f} "
      f"fine px ({(np.median(_sh[_ok]) if _ok.any() else 0)*TARGET_GSD_FINE_M*100:.1f} cm)")

def _heldout_rmse(src, dst, splits=5):
    rng = np.random.default_rng(0); vals = []
    for _ in range(splits):
        idx = rng.permutation(len(src)); cut = int(0.7 * len(idx))
        tr, te = idx[:cut], idx[cut:]
        if len(tr) < 8 or len(te) < 4: continue
        Ht, _, _ = robust_fit(src[tr], dst[tr], NAC_FINE_SHAPE, verbose=False,
                              thresh=FIT_THRESH_PX,
                              min_minor_axis_px=FINE_MIN_MINOR_AXIS_PX)
        if Ht is None: continue
        vals.append(float(np.sqrt(np.mean(
            np.linalg.norm(apply_h(Ht, src[te]) - dst[te], axis=1) ** 2))))
    return float(np.mean(vals)) if vals else None

_before = _heldout_rmse(_src_in, _dst_in)
SUBPIX_APPLIED = False
H_subpix = H_fine
if _ok.sum() >= 12:
    _after = _heldout_rmse(_src_in[_ok], _dst_ref[_ok])
    print(f"  held-out RMSE  before {_before if _before is None else round(_before,4)} "
          f"-> after {_after if _after is None else round(_after,4)} fine px")
    if _before is not None and _after is not None and _after < _before:
        Hs, ms, mdl = robust_fit(_src_in[_ok], _dst_ref[_ok], NAC_FINE_SHAPE,
                                 thresh=FIT_THRESH_PX,
                                 min_minor_axis_px=FINE_MIN_MINOR_AXIS_PX)
        if Hs is not None and abs(np.linalg.det(Hs[:2, :2])) > 1e-12:
            H_subpix, SUBPIX_APPLIED = Hs, True
            print(f"  ACCEPTED: {mdl} refit on {int(ms.sum())} sub-pixel correspondences "
                  f"({_after*TARGET_GSD_FINE_M*100:.1f} cm held-out)")
    if not SUBPIX_APPLIED:
        print("  REJECTED — refinement did not improve held-out RMSE. Keeping H_fine.\n"
              "  With a large sun-angle gap this is a real outcome, not a failure: gradient\n"
              "  phase correlation needs the same edges to be visible in both images.")
else:
    print("  too few points refined to re-fit — keeping H_fine.")

# The correspondence table everything downstream exports
MATCH_SRC   = _src_in                       # ohrc fine px
MATCH_DST   = np.where(_ok[:, None], _dst_ref, _dst_in)
MATCH_OK    = _ok
MATCH_SCORE = fine_s[inlier_mask]
H_FINAL     = H_subpix
RESIDUAL_PX = np.linalg.norm(apply_h(H_FINAL, MATCH_SRC) - MATCH_DST, axis=1)
print(f"\nFinal model: {'sub-pixel refined' if SUBPIX_APPLIED else 'H_fine (unrefined)'}")
print(f"  {len(MATCH_SRC)} correspondences, residual mean {RESIDUAL_PX.mean():.3f} fine px "
      f"= {RESIDUAL_PX.mean()*TARGET_GSD_FINE_M*100:.1f} cm, median "
      f"{np.median(RESIDUAL_PX):.3f} px")


In [ ]:
# 21) CONSENSUS TIE POINTS — collapse redundant measurements of the same ground point
#
# SuperGlue is one-to-one WITHIN a call: its optimal-transport layer produces a partial
# assignment, so one OHRC keypoint claims at most one NAC keypoint. You cannot ask it for
# many-to-one, and within a single call you would not want to -- several NAC points binding
# to one OHRC point means the descriptors cannot discriminate, which is repetitive terrain,
# not extra evidence.
#
# The redundancy that IS real comes from ACROSS calls. Tiles overlap by TILE_OVERLAP, coarse
# blocks overlap too, and the block cross-product matches the same ground several times. So
# the pooled set already holds several independent measurements of the same tie point, sitting
# as near-duplicate rows that the fit treats as unrelated.
#
# Collapsing them buys two things:
#   * averaging N independent measurements cuts the noise by ~sqrt(N)
#   * the SPREAD inside a group is a direct, empirical uncertainty for that tie point --
#     far more informative than one global RMSE, and it catches contradictory matches that
#     RANSAC can miss because each one is individually plausible
from sklearn.cluster import DBSCAN

CLUSTER_RADIUS_PX  = 3.0    # fine px: two matches this close in OHRC are the same ground point
MIN_AGREE          = 2      # measurements needed before a group is called a consensus point
MAX_DISAGREE_PX    = 1.5    # spread of the displacement vectors inside a group
KEEP_SINGLETONS    = True   # keep once-seen points, flagged, so coverage is not thrown away

_disp = MATCH_DST - MATCH_SRC
_lab = DBSCAN(eps=CLUSTER_RADIUS_PX, min_samples=1).fit(MATCH_SRC).labels_
print(f"{len(MATCH_SRC)} correspondences -> {_lab.max()+1} distinct ground points "
      f"(radius {CLUSTER_RADIUS_PX} fine px = {CLUSTER_RADIUS_PX*TARGET_GSD_FINE_M:.2f} m)")

con_src, con_dst, con_n, con_sig, con_score = [], [], [], [], []
n_dropped = 0
for lb in range(_lab.max() + 1):
    sel = np.where(_lab == lb)[0]
    if len(sel) == 1:
        if KEEP_SINGLETONS:
            con_src.append(MATCH_SRC[sel[0]]); con_dst.append(MATCH_DST[sel[0]])
            con_n.append(1); con_sig.append(np.nan); con_score.append(MATCH_SCORE[sel[0]])
        continue
    d = _disp[sel]
    spread = float(np.sqrt(np.mean(np.sum((d - d.mean(0)) ** 2, axis=1))))
    if spread > MAX_DISAGREE_PX:
        n_dropped += len(sel)          # the measurements contradict each other
        continue
    w = np.clip(MATCH_SCORE[sel], 1e-3, None); w = w / w.sum()
    con_src.append((MATCH_SRC[sel] * w[:, None]).sum(0))
    con_dst.append((MATCH_DST[sel] * w[:, None]).sum(0))
    con_n.append(len(sel))
    con_sig.append(spread / np.sqrt(len(sel)))   # standard error of the mean displacement
    con_score.append(float(MATCH_SCORE[sel].mean()))

CON_SRC   = np.asarray(con_src, float).reshape(-1, 2)
CON_DST   = np.asarray(con_dst, float).reshape(-1, 2)
CON_N     = np.asarray(con_n, int)
CON_SIGMA = np.asarray(con_sig, float)
CON_SCORE = np.asarray(con_score, float)

_multi = CON_N >= MIN_AGREE
print(f"  {int(_multi.sum())} points seen {MIN_AGREE}+ times "
      f"(max {int(CON_N.max()) if len(CON_N) else 0} measurements on one point)")
print(f"  {int((~_multi).sum())} seen once{' (kept, flagged)' if KEEP_SINGLETONS else ' (dropped)'}")
print(f"  {n_dropped} measurements dropped: their group disagreed by more than "
      f"{MAX_DISAGREE_PX} px")
if _multi.any():
    print(f"  median standard error on a consensus point: "
          f"{np.nanmedian(CON_SIGMA[_multi]):.3f} fine px "
          f"({np.nanmedian(CON_SIGMA[_multi])*TARGET_GSD_FINE_M*100:.1f} cm)")

# Does consensus actually fit better? Same held-out test as everywhere else.
_before = _heldout_rmse(MATCH_SRC, MATCH_DST)
_after  = _heldout_rmse(CON_SRC, CON_DST) if len(CON_SRC) >= 12 else None
print(f"\nheld-out RMSE  raw {None if _before is None else round(_before,4)} "
      f"-> consensus {None if _after is None else round(_after,4)} fine px")

CONSENSUS_APPLIED = False
if _before is not None and _after is not None and _after < _before:
    Hc, mc, mdl = robust_fit(CON_SRC, CON_DST, NAC_FINE_SHAPE, thresh=FIT_THRESH_PX,
                             min_minor_axis_px=FINE_MIN_MINOR_AXIS_PX)
    if Hc is not None and abs(np.linalg.det(Hc[:2, :2])) > 1e-12:
        H_FINAL, CONSENSUS_APPLIED = Hc, True
        MATCH_SRC, MATCH_DST = CON_SRC, CON_DST
        MATCH_SCORE = CON_SCORE
        MATCH_OK = CON_N >= MIN_AGREE
        RESIDUAL_PX = np.linalg.norm(apply_h(H_FINAL, MATCH_SRC) - MATCH_DST, axis=1)
        print(f"  ACCEPTED: {mdl} refit on {len(CON_SRC)} consensus points "
              f"({int(mc.sum())} inliers)")
if not CONSENSUS_APPLIED:
    print("  REJECTED — consensus did not improve held-out RMSE. Keeping the raw matches.\n"
          "  That happens when redundancy is low (little tile overlap) or when the repeats\n"
          "  are correlated rather than independent, in which case averaging buys nothing.")

CONSENSUS_N     = CON_N     if CONSENSUS_APPLIED else np.ones(len(MATCH_SRC), int)
CONSENSUS_SIGMA = CON_SIGMA if CONSENSUS_APPLIED else np.full(len(MATCH_SRC), np.nan)
print(f"\nCarrying {len(MATCH_SRC)} tie points forward, residual mean "
      f"{RESIDUAL_PX.mean():.3f} fine px = {RESIDUAL_PX.mean()*TARGET_GSD_FINE_M*100:.1f} cm")


In [ ]:
# 20) FINAL OUTPUTS — correspondence table + warped raster
#
# Two deliverables:
#   1. every matched point, in fine px, native px and lat/lon, with its residual
#   2. the OHRC image actually resampled into the NAC frame at full fine resolution
#
# The warp is written in row bands so a 25 km strip never has to exist twice in memory.
import csv

H_native_final = np.linalg.inv(M_nac_native_to_fine) @ H_FINAL @ M_ohrc_native_to_fine

# ---------------------------------------------------------------- 1. correspondence table
_src_native = apply_h(np.linalg.inv(M_ohrc_native_to_fine), MATCH_SRC)
_dst_native = apply_h(np.linalg.inv(M_nac_native_to_fine),  MATCH_DST)

rows = []
for i in range(len(MATCH_SRC)):
    o_col, o_row = float(_src_native[i][0]), float(_src_native[i][1])
    n_col, n_row = float(_dst_native[i][0]), float(_dst_native[i][1])
    o_lat, o_lon = ohrc_fwd(o_row, o_col)
    n_lat, n_lon = nac_fwd(n_row, n_col)
    rows.append(dict(
        idx=i,
        ohrc_row_native=round(o_row, 3), ohrc_col_native=round(o_col, 3),
        nac_row_native=round(n_row, 3),  nac_col_native=round(n_col, 3),
        ohrc_row_fine=round(float(MATCH_SRC[i][1]), 3),
        ohrc_col_fine=round(float(MATCH_SRC[i][0]), 3),
        nac_row_fine=round(float(MATCH_DST[i][1]), 3),
        nac_col_fine=round(float(MATCH_DST[i][0]), 3),
        ohrc_lat=round(float(o_lat), 7), ohrc_lon=round(float(o_lon) % 360.0, 7),
        nac_lat=round(float(n_lat), 7),  nac_lon=round(float(n_lon) % 360.0, 7),
        match_score=round(float(MATCH_SCORE[i]), 4),
        subpixel_refined=bool(MATCH_OK[i]),
        n_measurements=int(CONSENSUS_N[i]),
        sigma_fine_px=(None if not np.isfinite(CONSENSUS_SIGMA[i])
                       else round(float(CONSENSUS_SIGMA[i]), 4)),
        residual_fine_px=round(float(RESIDUAL_PX[i]), 4),
        residual_m=round(float(RESIDUAL_PX[i] * TARGET_GSD_FINE_M), 4)))

csv_path = os.path.join(OUT_DIR, f"{PAIR_ID}_v4_matches.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
np.savez_compressed(os.path.join(OUT_DIR, f"{PAIR_ID}_v4_matches.npz"),
                    src_fine=MATCH_SRC, dst_fine=MATCH_DST,
                    src_native=_src_native, dst_native=_dst_native,
                    score=MATCH_SCORE, refined=MATCH_OK, residual_px=RESIDUAL_PX,
                    n_measurements=CONSENSUS_N, sigma_px=CONSENSUS_SIGMA,
                    H_fine=H_FINAL, H_native=H_native_final,
                    fine_gsd_m=TARGET_GSD_FINE_M)
print(f"{len(rows)} correspondences -> {csv_path}")
print(f"                            -> {csv_path[:-4]}.npz  (plus H_fine / H_native)")

# ---------------------------------------------------------------- 2. warped raster
WARP_BAND = 2048          # fine-level rows per band
out_h, out_w = NAC_FINE_SHAPE
print(f"\nResampling OHRC into the NAC frame at {TARGET_GSD_FINE_M:.3f} m/px "
      f"-> {out_h} x {out_w} px, in bands of {WARP_BAND} rows")

warped_full = np.zeros((out_h, out_w), np.uint8)
cover = 0
for y0 in range(0, out_h, WARP_BAND):
    y1 = min(out_h, y0 + WARP_BAND)
    # which OHRC fine region feeds this band?
    corners = np.array([[0, y0], [out_w, y0], [0, y1], [out_w, y1]], float)
    back = apply_h(np.linalg.inv(H_FINAL), corners)
    r0 = max(0, int(np.floor(back[:, 1].min())) - 4)
    c0 = max(0, int(np.floor(back[:, 0].min())) - 4)
    r1 = min(OHRC_FINE_SHAPE[0], int(np.ceil(back[:, 1].max())) + 4)
    c1 = min(OHRC_FINE_SHAPE[1], int(np.ceil(back[:, 0].max())) + 4)
    if r1 <= r0 or c1 <= c0:
        continue
    src, org = level_tile(ohrc_crop, OHRC_NATIVE_GSD_M, TARGET_GSD_FINE_M,
                          r0, c0, r1 - r0, c1 - c0)
    if src is None:
        continue
    H_band = mat_translate(0, -y0) @ H_FINAL @ mat_translate(org[0], org[1])
    band = cv2.warpPerspective(to_uint8(src), H_band.astype(np.float64),
                               (out_w, y1 - y0), flags=cv2.INTER_CUBIC, borderValue=0)
    warped_full[y0:y1] = band
    cover += int((band > 0).sum())
    print(f"  rows {y0}-{y1}  ({100.0*(band > 0).mean():.0f}% filled)")

warp_path = os.path.join(OUT_DIR, f"{PAIR_ID}_v4_OHRC_warped_to_NAC.png")
cv2.imwrite(warp_path, warped_full)
print(f"\nWarped OHRC -> {warp_path}  ({out_h}x{out_w}, "
      f"{100.0*cover/(out_h*out_w):.1f}% of the NAC frame covered)")

# georeferencing sidecar, so the warped product is not just pixels
_c = {}
for nm, (rr, cc) in (("UL", (0, 0)), ("UR", (0, out_w - 1)),
                     ("LL", (out_h - 1, 0)), ("LR", (out_h - 1, out_w - 1))):
    nat = apply_h(np.linalg.inv(M_nac_native_to_fine), [[cc, rr]])[0]
    la, lo = nac_fwd(nat[1], nat[0])
    _c[nm] = {"lat": round(float(la), 7), "lon": round(float(lo) % 360.0, 7)}
side = {"pair_id": PAIR_ID, "warped_image": os.path.basename(warp_path),
        "frame": "LRO NAC " + nac_product_id, "source": "CH2 OHRC " + ohrc_product_id,
        "gsd_m": TARGET_GSD_FINE_M, "shape": [out_h, out_w], "corners_deg": _c,
        "H_ohrc_native_to_nac_native": H_native_final.tolist(),
        "subpixel_refined": bool(SUBPIX_APPLIED),
        "n_correspondences": int(len(MATCH_SRC)),
        "residual_mean_m": float(RESIDUAL_PX.mean() * TARGET_GSD_FINE_M)}
with open(os.path.join(OUT_DIR, f"{PAIR_ID}_v4_warped.json"), "w") as f:
    json.dump(side, f, indent=2)

# quick look: side by side + checkerboard, small enough to open
_k = max(1, int(np.ceil(max(out_h, out_w) / 1600)))
_wq = warped_full[::_k, ::_k]
_nq = to_uint8(level_view(nac_crop, NAC_NATIVE_GSD_M, TARGET_GSD_FINE_M * _k))
_nq = cv2.resize(_nq, (_wq.shape[1], _wq.shape[0]), interpolation=cv2.INTER_AREA)
_blk = max(16, min(_wq.shape) // 20)
_yy, _xx = np.mgrid[0:_wq.shape[0], 0:_wq.shape[1]]
_chk = np.where((((_yy // _blk) + (_xx // _blk)) % 2).astype(bool), _wq, _nq)
fig, ax = plt.subplots(1, 3, figsize=(16, 9))
ax[0].imshow(_wq, cmap='gray'); ax[0].set_title('OHRC warped into NAC frame')
ax[1].imshow(_nq, cmap='gray'); ax[1].set_title('NAC reference')
ax[2].imshow(_chk, cmap='gray'); ax[2].set_title('Checkerboard — features must run continuously')
for a in ax: a.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'{PAIR_ID}_v4_06_final_quicklook.png'), dpi=150)
plt.show()

print(f"\nFinal outputs in {OUT_DIR}:")
for f in sorted(os.listdir(OUT_DIR)):
    if f.startswith(PAIR_ID) and "v4" in f:
        print(f"  {f}  ({os.path.getsize(os.path.join(OUT_DIR, f))/1e6:.1f} MB)")


In [ ]:
# 22) RUN REPORT — every stage's result in one place
#
# Gathers what each cell produced and prints it as one readable block, then saves it next to
# the other outputs. Every lookup is defensive: a stage you did not run prints "(not run)"
# rather than raising, so this works on a partial pipeline too.
import io, datetime

_R = io.StringIO()
def _p(*a):
    line = " ".join(str(x) for x in a)
    print(line); _R.write(line + "\n")

def _g(name, default=None):
    return globals().get(name, default)

def _has(*names):
    return all(n in globals() and globals()[n] is not None for n in names)

def _fmt(v, spec="", missing="(not run)"):
    if v is None:
        return missing
    try:
        return format(v, spec)
    except (TypeError, ValueError):
        return str(v)

def _hdr(t):
    _p("\n" + "=" * 78); _p(t); _p("=" * 78)

_p("=" * 78)
_p(f"SIH26166 — OHRC <-> LRO NAC REGISTRATION · RUN REPORT")
_p(f"generated {datetime.datetime.now():%Y-%m-%d %H:%M}")
_p("=" * 78)

# ---------------------------------------------------------------- 1. inputs
_hdr("1. INPUTS")
_p(f"  pair id        : {_g('PAIR_ID')}")
_p(f"  OHRC product   : {_g('ohrc_product_id')}")
_p(f"  NAC product    : {_g('nac_product_id')}   corners from {_g('NAC_CORNERS_SOURCE')}")
if _has('ohrc_shape', 'OHRC_NATIVE_GSD_M'):
    _p(f"  OHRC raster    : {ohrc_shape[0]:,} x {ohrc_shape[1]:,} px @ "
       f"{OHRC_NATIVE_GSD_M:.4f} m/px")
if _has('nac_shape', 'NAC_NATIVE_GSD_M'):
    _p(f"  NAC  raster    : {nac_shape[0]:,} x {nac_shape[1]:,} px @ "
       f"{NAC_NATIVE_GSD_M:.4f} m/px   (source: {_g('NAC_GSD_SOURCE','?')})")
for _t, _c in (("OHRC", _g('ohrc_corners')), ("NAC ", _g('nac_corners'))):
    if _c:
        _la = [v['lat'] for v in _c.values()]; _lo = [v['lon'] for v in _c.values()]
        _al, _ac = ground_extent_km(min(_la), max(_la), min(_lo), max(_lo))
        _p(f"  {_t} footprint : lat {min(_la):.4f}..{max(_la):.4f}, "
           f"lon {min(_lo)%360:.4f}..{max(_lo)%360:.4f}  "
           f"({_al:.1f} x {_ac:.1f} km bbox)")

# ---------------------------------------------------------------- 2. overlap
_hdr("2. OVERLAP AND SCALE")
if _has('ohrc_win', 'nac_win'):
    _p(f"  OHRC crop      : rows {ohrc_win[0]}-{ohrc_win[1]}, cols {ohrc_win[2]}-{ohrc_win[3]}"
       f"   ({_fmt(_g('ohrc_kept'),'.1f')}% of the raster)")
    _p(f"  NAC  crop      : rows {nac_win[0]}-{nac_win[1]}, cols {nac_win[2]}-{nac_win[3]}"
       f"   ({_fmt(_g('nac_kept'),'.1f')}% of the raster)")
    _p(f"  polygon crop   : {'yes' if _g('USED_POLYGON_CROP') else 'no (axis-aligned box)'}")
_p(f"  fine level     : {_fmt(_g('TARGET_GSD_FINE_M'),'.3f')} m/px "
   f"(multiplier {_fmt(_g('FINE_GSD_MULTIPLIER'),'g')})")
_p(f"  coarse level   : {_fmt(_g('TARGET_GSD_COARSE_M'),'.3f')} m/px")
_p(f"  fine extents   : OHRC {_g('OHRC_FINE_SHAPE')}   NAC {_g('NAC_FINE_SHAPE')}")

# ---------------------------------------------------------------- 3. geometry
_hdr("3. GEOMETRIC PRIOR")
if _has('H_native_geo'):
    _A = H_native_geo[:2, :2]
    _p(f"  scale          : x {np.hypot(*_A[:,0]):.4f}, y {np.hypot(*_A[:,1]):.4f}"
       f"   (GSD ratio {OHRC_NATIVE_GSD_M/NAC_NATIVE_GSD_M:.4f})")
    _p(f"  determinant    : {np.linalg.det(_A):+.4f}"
       f"  -> {'MIRRORED' if np.linalg.det(_A) < 0 else 'not mirrored'}")
_g_ = _g('gate')
if _g_:
    _p(f"  pre-flight gate: {_g_.get('verdict')}  (margin x{_fmt(_g_.get('margin'),'.1f')}, "
       f"null {_fmt(_g_.get('floor'),'.3f')})")
_b = _g('_bands')
if _b:
    _ok = [r for r in _b if r['margin'] >= _g('DENSE_MIN_MARGIN', 2.0)]
    _p(f"  dense bands    : {len(_ok)}/{len(_b)} cleared the null")
    if _ok:
        _dx = [r['dx'] for r in _ok]; _dy = [r['dy'] for r in _ok]
        _p(f"                   dx {min(_dx):+d}..{max(_dx):+d}, dy {min(_dy):+d}..{max(_dy):+d}"
           f" coarse px   (spread {max(max(_dx)-min(_dx), max(_dy)-min(_dy))} px)")
    _p(f"  shift applied  : {'yes' if _g('DENSE_APPLIED') else 'no'}"
       f"   base {_g('PRIOR_SHIFT_BASE')}")
_p(f"  coarse model   : {_g('coarse_model')}")

# ---------------------------------------------------------------- 4. tiled matching
_hdr("4. TILED MATCHING")
_p(f"  tiles          : {_g('tiles_tried','?')} tried, {_g('tiles_used','?')} matched")
_p(f"    rejected     : {_g('tiles_skipped_texture','?')} no-texture, "
   f"{_g('tiles_skipped_overlap','?')} no-overlap, {_g('tiles_no_consensus','?')} no-consensus")
_p(f"  tile size      : {_g('TILE','?')} fine px, overlap {_fmt(_g('TILE_OVERLAP'),'.0%')}, "
   f"search margin {_g('SEARCH_MARGIN','?')} px")
if _has('fine_o'):
    _p(f"  correspondences: {len(fine_o)}")
if _has('fine_m'):
    for _n in sorted(set(np.asarray(fine_m).tolist())):
        _p(f"    {_n:12s} {int((np.asarray(fine_m) == _n).sum())}")
_ts_ = _g('tile_shift')
if _ts_:
    _ts_ = np.asarray(_ts_)
    _p(f"  prior residual : median ({np.median(_ts_[:,0]):+.1f}, {np.median(_ts_[:,1]):+.1f}) "
       f"fine px, spread ({_ts_[:,0].std():.1f}, {_ts_[:,1].std():.1f}), "
       f"max |shift| {np.abs(_ts_).max():.0f}")

# ---------------------------------------------------------------- 5. model
_hdr("5. MODEL")
_p(f"  model          : {_g('MODEL_NAME')}")
_p(f"  RANSAC thresh  : {_fmt(_g('FIT_THRESH_PX'),'.1f')} fine px"
   f"{' (data-driven)' if _g('AUTO_THRESHOLD') else ''}")
if _has('n_inliers', 'fine_o'):
    _p(f"  inliers        : {n_inliers} / {len(fine_o)} "
       f"({_fmt(_g('inlier_ratio'),'.1%')})")
_p(f"  hull coverage  : {_fmt(_g('coverage'),'.2%')} of the NAC frame")
_p(f"  trusted        : {_g('TRUSTED')}")
if _has('H_FINAL'):
    _p(f"  refinements    : ECC {'applied' if _g('ECC_APPLIED') else 'rejected/skipped'}"
       f" | sub-pixel {'applied' if _g('SUBPIX_APPLIED') else 'rejected'}"
       f" | consensus {'applied' if _g('CONSENSUS_APPLIED') else 'rejected'}")

# ---------------------------------------------------------------- 6. accuracy
_hdr("6. ACCURACY")
_m = _g('metrics', {}) or {}
_gsd = _g('TARGET_GSD_FINE_M', 1.0)
for _lbl, _k in (("in-sample RMSE ", 'rmse_px_in_sample'),
                 ("held-out RMSE  ", 'rmse_px_heldout')):
    _v = _m.get(_k)
    _p(f"  {_lbl}: {_fmt(_v,'.3f','n/a')} fine px"
       + (f" = {_v*_gsd:.2f} m" if isinstance(_v, (int, float)) else ""))
if _has('RESIDUAL_PX'):
    _p(f"  tie-point residual: mean {RESIDUAL_PX.mean():.3f} px = "
       f"{RESIDUAL_PX.mean()*_gsd:.2f} m, median {np.median(RESIDUAL_PX):.3f} px")
if _m.get('nmi_aligned') is not None:
    _p(f"  NMI            : {_m['nmi_aligned']:.4f} aligned vs "
       f"{_m['nmi_misaligned_baseline']:.4f} misaligned baseline")
_ts_ = _g('tile_shift')
if _ts_ is not None and len(_ts_):
    _sp = float(np.mean(np.asarray(_ts_).std(axis=0)))
    _gate = float(_g('TILE_CONSENSUS_PX', 6.0))
    _p(f"\n  between-tile spread {_sp:.1f} fine px = {_sp*_gsd:.1f} m, against within-tile")
    _p(f"  agreement <= {_gate:.0f} px (the consensus gate every match had to pass).")
    if _sp > 3 * _gate:
        _p(f"  -> x{_sp/_gate:.0f} larger between tiles than within one, so the residual is")
        _p( "     STRUCTURE, not matcher noise: relief parallax between two look angles and")
        _p( "     pushbroom geometry an affine corner model cannot capture. This is the")
        _p( "     accuracy floor, and removing it needs a DTM rather than a better matcher.")
    else:
        _p( "  -> comparable to the within-tile gate, so the tie points agree globally as")
        _p( "     well as locally. No evidence of unmodelled deformation at this scale.")

# ---------------------------------------------------------------- 7. outputs
_hdr("7. OUTPUT FILES")
try:
    _fs = sorted(f for f in os.listdir(OUT_DIR) if f.startswith(PAIR_ID) and 'v4' in f)
    for _f in _fs:
        _p(f"  {_f:52s} {os.path.getsize(os.path.join(OUT_DIR,_f))/1e6:7.2f} MB")
    if not _fs:
        _p("  (none yet — run the export cell)")
except Exception as _e:
    _p(f"  (could not list {_g('OUT_DIR')}: {_e})")

_p("\n" + "=" * 78)
_path = os.path.join(_g('OUT_DIR', '.'), f"{_g('PAIR_ID','run')}_v4_REPORT.txt")
try:
    with open(_path, "w") as _f:
        _f.write(_R.getvalue())
    print(f"Report saved -> {_path}")
except Exception as _e:
    print(f"(could not save the report: {_e})")
